# glcuda vs llama.cpp - the comparison the 5.85x figure should have been

The number this sprint was framed by is not an engine comparison. The
2026-08-21 notebook sets **no optimisation flags**: it measured glcuda's untuned
default path at **1666 tok/s** against llama.cpp's **9740**, and called that
5.85x. In the same era, with the production stack, Wave 11 was already at
**7397** - so the comparison put llama.cpp's tuned path against ours with the
tuning switched off.

Three things are fixed here:

1. **glcuda runs its production stack** (`FORCE_Q8`, `GRID2D`, `FUSE_Q8_GLUE`,
   `NTILE128`, `BSTAGE`), and the run asserts that from the dispatch line the
   engine prints rather than trusting the environment.
2. **The tools are interleaved and position-balanced** over four repeats. The
   old run measured them one after the other on a machine that drifts ~8%.
3. **The remaining asymmetry is stated in the table**: llama.cpp runs native
   Q4_K, glcuda repacks to Q8_0 - twice the weight bytes. Not corrected,
   because correcting it would stop this being a comparison of what each engine
   ships.

Everything else is the machinery that already works: the wave bootstrap's patch
stack and on-device parity gate, so the engine under test is the current tree
rather than whatever is committed; and the llama.cpp build cell lifted from the
old notebook, which already paid for every trap - no prebuilt CUDA Linux
binaries, `-j1` or nvcc OOMs, `CUDA::cuda_driver` needing libcuda.so via
ldconfig, and `-DLLAMA_BUILD_SERVER=OFF` silently deleting llama-cli.

⚠️ This build takes ~30 minutes before a single number appears.


## 1 - Bootstrap: patch stack, parity gate, build glbench


In [ ]:
import ast
import base64
import datetime as dt
import gzip
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import statistics
import subprocess
import traceback
import time
import urllib.request
import glob
import zipfile

NOTEBOOK_BUILD = "h2h-llamacpp-v1"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "3bce8dd7b8aaa2765855ab927c611b54981f9241"
WAVE3_PATCH_SHA256 = "5f09f6147636c36db4e23b9d5f16a3384ca4c0b69679d5443d7c9a396387a508"
WAVE3_PATCH_GZIP_B64 = """H4sIAAAAAAACCu19eXPbSJLv//oU1dqwmzRJiDh4ia3Zlm11j8NHeyTt80zoaSmQBEm0cNAAKInb44n3Id4nfJ/k5VG4CFAG17MRzVE7ukUKqEpUZWX+8qgScmrPZqLVmtuRMI/mzmQ1NY/CYHJ0awWe5YTy0ih0ex1lGT2IcYVGB7Y3tR5Ee2zqE9VUlP50PB1Mp0Jtt7uGcdBqtSo966DRaFR73o8/ipbWbfZEg37++OOBODoS1p0VrMW9GSzhRygCqxVY5tT25iJaWOLsw+Wb8zNhTiL7zoxs3xOhY47FLPBd8U4T8LsdhcK/90SLqM38gPrNzcgSP5+9f98Umt7tE/1QPAhNG4i3L8WJ+EfHaIv3L4U/QzpRYM5m9kQsrUA45sqbLBJqZtzHNaPAflDEqeOImCD0NsXY8Se3IlyYgUXPDk0Xvvi3lhc2RejjtYMWUAPirdsWt85NyJ5YotY1RODf4xh1TeAINXhoXZjelKYIt2e6dtDYgY7RJzp6GR2cXbbvxHSsUOAMPP9eXFye/nz2Wrz5IC7+fHoO396fvf/l/G/AbngC8kg+3JsSoYnvhSvXmorxOrOcx+I04es08IFb/Yem+OmnD7QuoZj74vX56fvW2F95UwUJkYToJCF6RkLw31/6o7a48E/F51CEEUiIK95cAGftENZr7a8iUfuE82655q+waCcn4uV/XsLAHL5QVySl16LWf+jDsuhaXcT/TmA9eb2Aa7C0lj1fRMRF7vYRGB4tUCxB8My5a3nAyNo88FfLN6+ht2N61pHRFJE9l789M+rHtOZCmG24FrqWOzKvalFbNITsWH8BC90QC9OZvVC78A26vzCEojSMayFqvmeJFYySlvxRMrDOFcjw/MdI5tOVlyFwfXU7fnQookhm2m6KqQqk/taKbMdCXgkYVSMeUxNZHwrtBRBq8kdDBTLRvU+sT1a7ozZVVTTwQ4vX27HmplNXxEVkzhEHYI3nKzOYgoBFPjwKxKVf82DB6rRGpGOeD7IAbXGRQjG21r6UTVTIpTklQAE5BYGcRCxsuDzn1ueVHVi0oscCxegZCABID8zP9uAX4Av/hs8TP5yg/tSQZrDyPFCEyWLl3YZEy4zEx/Ozn968ezd6eXr56s8CG9eH4mH0OTx6GEkVI910V2EEgxSn79798ur0EhRstYS50VJXHdJQWOZkIdatV5enYuKYLigYKLjlzJBLZkTEXB8eJBGhKe4XuFY7jAcZmON3zXoADnInBAtkN/AGlV7cB3YUWR6oGnLjHaHoMQiZPRUTy3ZqOJUjUD/xAphpT49ADOqIyJ2uYOUKuecFAun0WGhtArACwjWoy8sifCk05dyDa1uf3ORB0bIe4UIVxoLEzlIWg40BvAc16BotAguyRAo9R1mfqCimKBh+YIPUmo60I0wnnpPe7mk7zgnXoPXP+3cglDs7tMcgCAqIGGD13BnNLdcdua45+tyvHZDCK0szAIxVViA8y9H957BJ6trTEZx7RlM1SFupaWDNhTIGsXwW/GD0/zTMXp7h5dkPupa/PAayz4LpD4YB11t0neznVCimY889ASikjPsAeAB3KAjXQ2x0hCvFUJ0xlm/B7kRWeNCQTT6Zd5bQj4Vnok2C59lhZAWhmJgewISAB5i8VN7KHVv40AVaR9MBo0a8V5hUdmKj+XoUgQbyN28cfwNFHeZa88Tw1oPfjL+G6de1P0wG+uHsr5ew6LFNOZIqGfswoT+LwJBaraW9tBzbg4HethzfXxbHN0Yaba8pv6leblC0CPfhhBrA58btJbLg2dID3Y6vb18MlOCti0EKi2gLfZd2hF5UCTkjpvYQXoHYMzVJDodaUIIDwXScKYslSSXwU22KKxLO6yFJZ7/TBOM10JtaN5HOtA+to0ZdbO96WHpbp9sICtigIFGuf8cu3gJw9XvwRKzABk0HREADSJ4prg/gNEAwA4MCgD42QzA/CTUkgCgjncQW+eHIMR/6oaCiNBKio9EihHLNWxAM8q0SMrYHjtBqQnwaWw4OAEhaD8A+Z52HIny8Y7VogKkRTMbTRuwywQo4kb0EYACvFs0cDgPIBBYIpCX69diZhWeARpG1zdiGhFy6fK1NwwvK6JowbhGu0C20Qeiljw1CO0f7btqOALSVgwN+y4VJ9G8CTYCtUlDDhVOqoPitO0wcPDnLEyF7gyHoGvIRK0e5t6eWfE5Gc1OK2lAyPOqDkONS+7NZaEVMAeaohCyQejP9yYSScQaZiTB8gCh2MkOMx0kizw52SL42PXH7YMONwfIDsvxJISnMQ5I2zDwXNY+f/cgEjWb6k6kMt49svTEytXxQ6yw4lgwKrPdyFT02qk4z/ZnD2HA1VsKE7SB8TdbxeFTx4M2HYrPka9LK9mQrPXe/aww315DgEBwGDokoSgGhPmgw5qRSjYwEh0R5kGiUiglNxiiREKKP4dXInnInUFOpA13ZSVeHJZ0wLiGYVI0BWnG115Mx1gY7NVqtab/Jv8ixTe4ixN14gTWVZLg33NY9/siMBQYxd/wxAhK49uCXRwGgzPTOhLAyhJhPO7od19kdSBVbo3FoHeRFKmuOHwN6ctvoZ8eSvdnPDTVlvzZoklHb1m/AvYebGsrWjOcAnWASMxsAvM7c7XabA+DuwICPHHdZdohvukY/1S2kH8JJnrA0gDIWpWSJbHp+dvpagIGxIFiIHQkEcIrI3Bf9TKCobLJWJ0FWtwBRHN++EJh3KDA+7WwUOJR2NfoFBlA/2bu72VOGnjzSzU68ILpe4BqExeKUghHJNheTBABH6FbXM3jxbcT0vlGPFTXhoRGzQSuIEd+Ta22UPghXuvRRGCOzPIE/A16NBgon3ZqCGLcRpETZP3jM7VjU3sqEDXka9Yxj8zGwXUu8+o/z87MP21zR2/FJW9zZJpl+clkTl1oRlwsrIWY9LCGYsRGqI0y/3VrWkh1taGxO162QLrOHNIWn2PAQCulhlY4pnE2dGyDPLiB4pzMLXEkYze24ocogFgLNEMY2Ac8kjPNPcD/jN+T94gTEc3fUjTtZR7k9a8t/JQ3UsgZgmpbKPOJVIXeaRMNIn/Hjd3xdjCGIfv/+dPT23S+/fIxvPluqKjqkjJFMJpnAFfq76vVX26pJ24ba3docPPpnC1W21PPNEOhhnsoMG8VRw0J9lJQmSRmPk1KJlLa5FoHWbSZTLd7sFcKaZClmWlOOsXBHz8Q6Bymzj6UC01pJd2XJQM9OiiHVGGYwSNbp0/mbyzMOMjRVbao6KGRn0FQ7oJDUgDKmo5en58cJUmO0TblaguuJ74IXY4laeGsvlwDc9wsf/bw1+jctf9YKTA/MCaWV64Ij9VhkgJOJwFz87cMrDE1acWjCycoE+hsyZjoGPTMxDvvlw6uzJqjsKswmaF3O21HcIO3ChjTRosRSlwWUl5QnFG1RuxVtRQEegLkhfKlvIdTLieQmIRUJqV1F0dUigYKgtmJfJJErkoFFqcsD63/lTdqY4bwuJ50V3BLSOsvrVtIqk07hlLFqW2Q/5lCK8Sre4pisggBXjliobBoroxfLppqFGWcTZrgdSm+sfwQ0j8MJuFzXFTpkMAUzMVs75BerUaD9VWQpJZgs0dcJJviS+kvnK89DTrO71AKb64P4o8kF+4V6AexEGyQVos1Z1dt4NTZMrd4pcRoSmTilBSdTzlBhaLTbNdDI0S4ARR4q5AjUTeXf1P1N14iHBD/Big4Lvk5yF5yXooNCEQP+1LU0IcKsipXXIPbrnettDTqyAdsb3uSjeevqLvPWNuat7eG8+zxvfZd56xvz1vdv3nqb593ZZd7GxryNPZy3xvPu7TLvzsa8O3s4b8Y1fSdc627Mu7uH82ZcM3bCtd7GvHt7OO8+haAGBM5qu2ziyWxS7/cM3UzcLqtRPhiT0bhJWxeuZUWhwLRy7A7RVgyac86o+NAT9xItaX3HZqCEa29SjOH+p+OGzcyWSrkb+tC1zXCeZiAD7WJPzgnjh1ZClxOq+KExxw1A1D5wvNNNJY1iEMlg15xiJiaXUuLwhX7EmbDNrGycdmoKozgKvjlVOW+HAsTpoih2x5A/V3TrGprM1PYjLRoGt0mc1qTNnZYn9BtS4rZfhixKn9TjgmS34znGc8tHcGo75YEueQjr1EGx7RpNY5BwUf09sVCrwEK9Egs1bvtl+I2ci2NfUPQYp17/8uFs+DtimlGBaZ1KTDO47VNgWrcC03qVmNbltk+Baf0KTBtUYlqf2z4BpmkVzIJWySxQQg7aPgWmVTAEWiVDoGnc9ikwrYIh0CoZAs3gtl8ocxRPVToKgRVJh4JPP3XUQeqUyXa458sZVnMS+GHI5+foUFdyRNEUHVWTR8fo2CCedeazrKE8cxrykcV7PBKGxztwY8ifCU7xRrZrhYp4Jc9S0NnAZ/2TEzoX+EzX6BsexfjhBJ7bJFL5s37yhFPxHKUifj77cHZOJ/9qlmtHI9ySUZbrevb8sq61Zr4zFSsv8B0Hz4wsA//OHDuYT5uvHPDTxXly2uofEB3yHtdAU/FsW+tP4h98gPm/S0jVdVVLKOH8+JjE0cV7UftHp/1M+JPJaml6k3V9SGHF2PImC3GF59tay4UZWuNrcXr0UkytCUglsH1h4QkXeVIUBuJZUWtsmRHvW/U5hOMD7XjYHC++/MRHltOjzP/8Q3okbgPapu8YnYK4ff30l3j89JfYcvrra0fx+uogPf1FpyUfO4v3CCFV0/p8qK+M0O7nyNS2ZuQG9k0HyTqDHu7hdw29OShhfSYfTKIJAxj+sUX/dLfoe5pKx2K7vVRTs5uNm1ZQo9kZtKeTYywdxP/agRo9TdzE+ySyHad2oGG8mZXywOBHlsgRH1ciERI1KbF0mpFmWS8Rq5RWXqwSSqC+ZZQ2WC/J8M/OcFPIEEfiA1+lPVk6jJgdYODjPBaw4wrvXDPDjKE8u9yjVerrm9m70/YfC/V7WqgOnzIfDAoLpf6xUL+nheqSh9JXO4WF0v5YqN/RQvX5j+r6/W/aiG7v4VZ0nzLojUH7W7aiwbncw5n3eObat2xGq+093I7uD3jmxrdsR6vtPdyQHqg88+63bEir7T3ckh4wwg3637Ilrbb3cFN6wAinttvfsiuttvdwX3rQk1PfCeP6han393DqAzn1nUBuUJj6YP+mDn6InPtOMKe2C6dq2/s4eV1OfjdfruDMqeo+Tl5inboT1qkFf07V9nHyEu3UndBOLbh0qr6Pk5d4p+6Ed2rBq1P30KsDXZWT3w3wCo6d2tnHyUvAU3cDvIJvp3b3cfIS8LTdAK/g3am9fZy8BDxtN8Ar+Hdqfx8nLwFP2w3wCh6euo8eniYBT9sJ8LSCh6fto4enScDTdgI8reDhafvo4WkS8PTdEnYFD0/bRw9Pk4Cn7wR4WsHD0/bRw9Mk4Ok7AZ5W8PC0ffTwdAl4+m6AV/DwtH308HQJePpugFfw8LR99PB0CXjGboBX8PC0ffTwdAl4xm6AV/DwtH308HQJeMZugFfw8LR99PAMCXjGToCnFzw8fR89PEMCnrET4OkFD0/fRw/P6KtNPKDXUDtar6lpxtP6+6Z/9pnsT7+3v5nYiz9p+vTH3zT9K3BtL/6o6dMff9X0L8C1/fizpk9//F3TvwLX/kf+sOlfnmsVrIFWyRrQawWg7ZPgWgVroFWyBvTSTGj7FLimV7AGeiVrQEOHtk+CaxWsgV7JGtCToe2T4FoFa6BXsgb0x07Q9klwrYI10CtZA0qEQNsnwbUK1kCvZA30Prd9ClwzKlgDo5I1MNrc9klwrYI1MCpZA0Pjtk+CaxWsgVHJGhgGt30SXKtgDYxK1sDoctsnwbUK1sCoZA2MPrd9ClzrVLAGnUrWoNPmtk+CaxWsQaeSNeho3PZJcK2CNehUsgYdg9s+Ca5VsAadStag0+W2T4JrFaxBp5I16PS57VPgWreCNehWsgbdNrd9ElyrYA26laxBV+O2T4JrFaxBt5I16Brc9klwrYI16FayBt0ut30SXKtgDbqVrEG3z22fAtd6FaxBr5I16LW57ZPgWgVr0KtkDXoat03eNC3f6lb2ysepPZuJVmtuR8IsK2/v+lMlCMV4+70DqrcltJk+0bq6oljjabc96wi13e4axgGe7XuE8kGj0XiUOr0ksEtvZu/KF7MvV2PBVUHFW2p8YUXit/g44VFcqdT3LHzDIr5dEiu6mxEVCxKzwHfFzc/vXv3H69PRudbp3gyxpmja/eomoXp8jK+IHFmeOXas6c21fFk7l/Ia+77TjKviHIlP/JZF+UpFN/dSxXDhr5ypmOBb3/EFlSKtrRxivWt8hbzWep0Sk2WVM8PPDfvn8zevtddy4I3SgWNhVm2aHTq148vx4Gk2sxFI7bFkZXIttJ3VCAS+cCPwl1Z6kf+MS6NigCosEJ30tN2lU1wa/GfPiHnZS/jPWga2Fzned7XDKxaFa26HddFs4Be+gVLIuYga3KEVjmvY40tJ64f1YUr0C882V9kWSyNP7akZWVjRzQ4zpdvMwI4WrhXZk3jFkP1Yq2qKlSdzxAJr6ZhYynFriVxePC4Bdw+EkyK5b+jt/+swR89fRi0QgZUHAsOF3AN/Kivezh1+t+ccS8rBg+mOhRW6xaWhpGQc4DKvrDgRrmsqdjgKfdeq1cXz5/DM6fGx5d0dH9+ZwcgPa4c5MTqsp82HKU1YKknyt/TitsUC4ZV1fqkiedmyHWaJf0mX6pfbWqmoEOC6ZjN/JaC3vubGw6PcaCeFGkBk5VhgFqLRDBYFmVo7nDt4cwSweVj/90K/VPC3dY5bbKPAGrKtN95NetJ7ODtU0bnT7TVV7XH1wcL3CrLgIOZiqv0XjDEx68PV2EVQd3KVm7Nog/ZAyuq/A9B8XtlBWl+cCziH7qjXkTOhgs0bAITUqFayd2cHvofv/JRSiRg980QehWrPcQJ1fLUt4k9WsGhm3JovfokLyeJYXppYG3HK8nTztyt8n28TXw18DfL+V/mr7V2LH8WnK7hMv/zn5U38Nts3Hy77KbVcWcXae01RxaUZ3oqXTZ5wo66IC9O1QDWtAGYdCjPMmocLGCsALJaT/9wftUF1zJtr8f/+z/+lZwGrW675KzzgL3BTXPin4nMo+BXIVILawfdsrnnpwS0Em9bo9GPbVr72NGpUrzD7xmSyC1g8uS9q+GTHMUH1J8ulWECLVuS38BONz701TQklqCnfy4yFw+5lsfhYeiaW7cD1upJ0i2vxHSWiIm6A06Op7YpnMIKTE9G+aYob25OXwCmJr1F17x9OQBBvaKQprWDleTABelN0iCb65uP52U9v3r0bvTy9fPXnm3qTxQ5f8XxzdBO/5PkmfctzSgvLeNKjbvi1zyAv/ILqfNFvYJb1EIETSK/hNQOL3nvaFB7WDEypYXEWEBQQhbMg8MEFAjxEDkllwLco+1FcfhDXnjDa91IF2plPiaXB3o9MufJMU2qPTDkz01OCBUZwM4hsxCxsP7ahc1qU/Sh+OFWAR/dvaXlY6jRGmleXp+HOfCso2MIMR2AAEq/r366QA/e1iWMvl+vj48j3R67prUdmMF8h9IT1a24Zgw9qKFAALa2xuvX7+Ccp3bYsOFOubIRP51YInPyhBgL4s0Mz+VMWiKfWeDUfmSEY/mhkff6ullnipgCX/zDzcOC2lIRNQcj5LAWaGRF5hCaX2NS11ttcVXUk3Soj/V0tVccNmuSfkhq64NfAGNmLBZCEkCZ9d3U9Z8pLaP9p+2jRC7dMII5GKCGZ4wM6M7VRU8yaYlQHfEfTkDexClDduAKyAtBV4+rU3XYHCwt1NV0WL91uUfHfc3cVCRtAXrzAb6Pk22R059ubngW19qq0vs6sQPwC8AVqXgiz6nMVVPomi78zrwmiSFER2mWFxoQOulsKG+3aC+BQDZEaxOQulkAs1Q5CCzGpCh/oKie/wKLQ4Omt4GE954V+w+Cke7vdSZbeMfrK5ISQZzwkFEtFUYCzAuBgPZiTyFkLNePdZuecd/2AA/kL29iRXGZXgVlU3+ycYVb+Tnvj9wwbM3fqqVeWgNlHfB+/eCmNbotCFXJjjjPVoSey2gFCeM69YO25uY5fCTZAR1FtG0azo6JcA6iCNxNGYVakMwjy8fKvo4v3vY7ikgMV1r7/7fu6MgGbESGyFW9/SW9nVTLW7qQDjtgEV6R2qEQAv6BT5DkdVu2FMQqWO+OX3FtTxe17/Vu1q4BsQTMHUzFK2Kf/0FXOAk4a0GUijsAamyF6Mljs15tjiARSFPll4biSF1UUfemUxQgYl7RACwddwO6syQ9xVyGHcGBpVTCVMkzPB2IyZRAFa9CfeO55+VFmYDWBd3d2aINjLBRuDtFBZtmpPEXtsL7R1XpYWpOodkjjpn45OMYR4Gv4RzKQPRHP4zFcKUo6tuthyahL+qQ9FCXbJyNlmcclknT4bAJLDbBwmJE3tV5OIPPsxwm0iwTyj89ImO3F+TxMwI3mazAQBAcFaaIqfgJXE2zS0rQDctOnvwKAgU+D695v0duKpbBSoZOxD4G9zFXlqJGMBBYOg4zcHawWIF0YYRVARE0pi0jX88lmmwE/ltqESimPCrp6uJkDzDIKyy3oWhm3vivRxtKcJLMpN7VTYUhOnJKfR1UrkFnEDKA3c+xJ1JoFlkXojrrHQSbwwTWXOWI0fzNaBaxdzNqkm3QYpuhSgvtJVROpwjGzSFwurBy1yHKXETbQuzxCHltoRnY4sy2seWJTem6J8XcQrZUia8o4g2U89HYvZkfVLlz5o6zP9sVM326d0xlYSWNTwdH8jE0MmU7E/7Imx8eYZhpNzKU5saN1Lb/yyFIqhDECnAShbCtKfzOnQ2y35/K2sXmbK1TC85TlKlzUarWEHr0MnIpl4Ne6OMIfzzZkbyMjl/nKRCEaj0YrCGwxS5BLQvF9CBVWy1p9G/pgE8fykFvoKdMKJDLH+hj5K5AGrgqPHSh2PtzkEso0c4BzlAUWoWSN+J3l2PAKtx9gicBr0IzrMqZlxllL3p3eyNKpy5DhcX7lZP1lBnooiQnWyDNdUJaNiuykY8u4XPvtuKHKaqR5A+hHJqkOemCeNJlxvXZU3zQNEnG258EOqcPYDALbCqoC1qPFfLIyr21Z6jyHyk0rxwfxI79eGX7TvlL/eCT5W1rG1Stxc0qQ9fv/3f6+Xrlt8D3M/fDV+bufxD3l8saIg7+C/YClHa/FMnoww1yoBLE1wsEF5m4d27PqGY9DwQthrQ6+BUTHAUTZ8J38jr9jdOXU/y6+czDza4YT2wZ/D+R3224Qp2kKG0HJZbkHNB4P9IFqKEq7PxkY2mzrHlDasbD9k97i4vL0Egn+gAsc1d+N7kcg1rUDcrEpl29zqMHBsM+7DN+jFb8zPSy4SzabQ4xcO/wF62SFtu8pTO8T58diJUoTJmnCDaOYpqwaJkhYuNUNxhk17wjiixtK53C2RF7FGkA3Q1DWyAZoskwvZOcTgACdTg/UGrUKldbxQ07zyzILruX6wRoLlB2JGpUaowpUwpyj7ERC09sY0twv7AkaYowIOA1Aw714r2BJMujKxcXyfXVd6z/el/J/NJXXlmOPSZjAZoP3kpQiAwMcuIo49dCHtmYze2IjfLA5B9xIBNr27uCGJVkDlHjPzPSQI+EKJjtEsTaz2yBYzAyzmJJfwBxyBUDq76nEsjkGt53lZaBhiIQfwKNEYNwRevO1g83CGBg7IYuXVlgnnyJNn08CUzo3uYxkUumuSIudHRA6lJUWFvbCUQKe9rN1vygDfG+uKRNPJdZCJU8LZn+ryOQXb+BgR9TUOLkHpqPWr5fZm5LNrmPyQuWeFAQPJFlsCcG3SjciS4lFCzD083QHC1kkgwSKuIDUwxEnKI/WJC+vLk/zW2YZasBcgL87mRbwAxtcOgh/cTUdq0WpAbQ3Yx8iMXD6WiA96PwppcTeouGjcnbmQibEeV8WaUf3Pimk6VD8RrW5iemRXz4yE91QEO7/so44RicqxBzzAZ8RR38gwiCO30+BNG4wF6nR8m3se5SuldyDXwUedsjmKkubxpkQ0NTPIIH34aRJlQ/xJ69AU6ybUj5tD/DJa5ZT2nQ1NjbkMmyhN/N8ImDAhcG8BSfkQXbvUa1pzULgWuatPXkapXQ5dkU3AldETP0k70NpQYTBY4Zcy4Rnc/KEq7S3dUzt6Sp+PK7fjzz/0ry1JPZimgBQxBMUZAQ2jQJQaMXZKzMCUMN8HyWpSomBpKwJ0oFXFAjRwMnKcL1LthtoA4FzjP5z3yJZbJXTixPprGoxNq9Q3RDpJYCDxG+YhxJqMcpLg0i2A8eBuMuV6Wc2DNHFb5smY+fBoSnZNrjG1wfH1qni6ErXFUY3M13bsSGkPsR8JpgNh7ABoiq45TiUhI8tf7rvX0ous9c2tgjM10sLhb/06bIgTQ/ls9/vNdV+knr+ebl6708tZzPzjNX3ItrSMUPmW8xOO2QFwz0bf4auHCI6FntcUbz862o6t4qG4x68HXCN/BlmGH4rjhK9Rc8DL5ES2Cs8lVS2Lu/8OSUmk61CTAsT/M183IORzHPZtieDgihB2W6TNDC3jgXcDtaxCZwsMHk4ze1QS0+NbX251MCDKU/HrKNNYRY6cJKNBwxTPCoPq8QnGCBcAZsONrWUHtlHOWPSXRQxOaR1Cw1aDbz0OCJ6p4mFLf09s5TeBDALoiTw6iZRk7a7ZtY9kJSGE/e+wvqQ8Yd1WqYbS5iHC4ZsIYCDdYM1/q7EvICTcKtkzypJxwGvLIHTNc/baoNK2fxFQHBpbesCSFjsMSwXN8xVgC8EY/97GB3DhMFaNIV7LJ6jVpjR3+EhrbKHYMrHGtHCXoXRtWicSHnka7XnrnJfh4haXiQhpYvAc49y/MPy0X8j4WQ56iUzTum75iQh77G6dQ2g6irxLmDmktzq4yslZMuYi761Y65BsMAa0/kJ+g08RfB5atvCOdouOFoi8q1zAd3GDRnSdTqTbteEkG5qdLu9cfmxvs2uuaBu8yYd6DNUOtEHH1rOTZeZbxm/j6bW5xUgGkjMFLf3wCaAJaWpxU4LHYE5Stze49Ljdx5iyxLjrey5u7xvG4M9U/MxAsnuW+GWFbm/IQIxKi2XwM7YQwQKcujNYI0e2MRZTS0mB6EkwhwE+k7IDtRSnpzxfK+19AEdWv6shV4riAHW9sqdArN8Qk2Ah3+7QnZeHzQ2GCYB4et8a8RqSTmDGjuUt3VMHMyXK2jCah87pl+kCjHspEFJBhcyx8Eu3r75eAzh9B2W9saNmHt5ggj3+SFa5IM3LTx4I88D5FJw/Mxhcv4nFvJ0L4/VhAuA1ykDl/auqV2JLV0j/tKhL9k8Tg2TdbjxrbXb5dchSM9epz1BLLaW7ZHL9SXrgHJRe84sfc7nk4rDbqIXElr1dJp00ulIei8cjHJcY3lztPupeV15ID3Ikb/cW56mdFptpfNS1BA5cIRMhbasXGtq47kKow8wKAN4UrxOu6kaoHk9+NT/e5qHDzEdOiGQpCzG6FRlRHZsLcw7218FtPSmmJtLGDjWE08dbkrz01k2pChVdQYqwocu6T7+ugJpQcGAy/gUiBcgBlt5eBwStNGLOLqT6X1wK+R0WzFLyW/MZmnooJMPfOJ4lvYPgd03pXvHN3yE0/OZYLh0AE3pHO9bgckNOpVFPkt8zOeGQ8Ok/PoC1MDF9HOSbkL9YHLwWPAA+CBMrxu7uHS+B8/F3BzdrJY3TSzxQtem/r0H13z4/zNc1vjqLfx6d9NkisB8Uxjt1sV7PCIqTpO0SpLaiN1D3NVnig2giF5Tt/sMBykbMj1wR6Etj6bBo3H9O3KG71HN0zgndmAj378FM/FM5pyQxQ8tCqJzKxFYLm6RgX/sBxGI2jEyYMvMmdLm9FFIcizYnLxMhci0iU165YcSk8FXvCkeC7jZMAtDnCAgIjY4AWPN2D/NpMGYGqXhucuJmkP4JM0hUzBg400Qp5a/TKxFVmLjo1VgYMwlRhu4LhDxWROT0xImJTFvsucDIHSKjxAMWa0oicoE5dk63BXn7S93GYUKpU9k0l+m9eF5EIZbwcTGKeLBUV5WeQhILmfcj05umHyOZ7aCxmj+Huk+lEdBMNMSySWQ3jll4eTOO1nVlkwiSbvZQrsZQ0oAzILJQo/zs9N3o4s/n348uzgWVzWJ+dkPcLINPCF6xe5CjWCcftABEHLWP9ODfAyafj1GcWMdBI0GAAzknwXUVHzv10bPW+p5dwwyKPU2PgLqmQE4BDgreXhGioKLMeDB/weFdsE857AAAA=="""
WAVE4_PATCH_SHA256 = "8fd9de6b6e2a41b84e73835530b3018bf037e621a6110737bbdeb26139021bab"
WAVE4_PATCH_GZIP_B64 = """H4sIAAAAAAACCu1be3fbNpb/358C8Z40VEXRIkW9k05ebjeT5nFst8kejw8NkZDEEUUyBCXb03rPfIj5hPtJ9l4ApEiKUuKJp+3OWZ+mtAngArjPH+4FPX86Ja3WzE8JPZoF7sqjR+yaLuOA8aMJC925kXAy2dl04IceuyZuhw3brm0YQ9O0aHtIzHa7Z9sHrVZrD92DZrO5j/bTp6RldnpdvU+a6gmvpiFZUj/UGqT1HTlhfBWkj7WGTp5H14+9m5Dw1BuNWJJEyWh0jI/vviO/HJDsJ2ApiZNowsgT8itPo3hEVh3r12IX/FkYNE1DJ4muuCO6a+V2/PkG162TTzpZuDpZw79oleokdOaMelwn+HA8f0koxzl0EkfqLdBkibNY6wetKlExiKeJ7zGdcJcG8AjTaJETwTXrB807j1vSa4ezTxU6JRqNzZ+3483vR0fkA02WhK1ZckPWNPFpmBLtz6/OSJO41J0zDvyPgxUn6ZyRhNGALFgSssBQMuzbUobi+fUyvD2oysljbuQxIa6KoO5dSJ9jdFk0dxXLZvGNPxVEgJsw+A0YRhKF/t+YJloVd4fKQob3YyHTKCEO8UPSNgw/ZWD/9baxm+e/m3F8vVlUDKIog9vPiEMIw7Y7utkGaYhfzHsQB1hfCtOsZvN4lRrkRRTyFO0voCuYn0RglLhJMMTQZQFHwaEVeuDZWQKulHGjRAz4DP/8ECRLoumUgz9MIzHEna/CBVn6nhcwwiNp2p4TMFwjveHkv63hwKi40og7MAKcKdJtEk0T3D0iVoN8S+yG4GvPHm8kg6O4H84C5rg0pq6f3sBobUI5g/HZaPjVbCihjMu6iY4Ld3luCgHCf+2Le3TfZr1u4i7/Kf007+SysXeZOfWe+u6K2e0JL2F3+/fiJaqeQq261mF8qUT+RVK5u2S+UjpbEqpIaaekvBo4xhP3SEZTrl4ZcXq9wU317QqY0WnH6/d6AMz69mAwtGuB2Q4KJXS2ow+qltXTe6QJ/we/9/TpATFSmszQyJdOvw1/Us9LGOcOh12Sng1hqwlu6F3IlAtrYYNH+Jwm+GCzJQtB4jFNUj/1oxBeTm6Eg4oTNvWDgIAmQRdoEpTUskAPwY9xN0oKjmXaseQrDsoaBNGVJNbpkeeAU7yVi1SgR0JTwJwHTYNdgwKHxFCrMWjgz0JiE2MywA3lOnx+AS7tAP1p6/5+BL1Z4ADHHFj5iNyc+xek+YRcw9Mg5EfBrxGZgVISl/mBFh5Z3V6DXBN4YJhAQzDufVlCyv2urQ9Azv3uQB+iA3n984cT5+W7t8cjMWEhWohggFH1PL1AR65jNAGukVSKzuMYcgDwu2kgJTv1E54eqPCkRjZNSRNHcqL5KSfRVQjTClJX4AWuEh/JFdTDZR7YZ8bHxdrBLhKjKG0bwxNDn3jFrn2ekskK1pUwEiLARfjqkRZMveI0aBgHLaBVYryWYxgMVsh80xrkzCendMkyZY4DGqK7gtUJOlHiz/wQ4LFm9shr/3mmm80dGgnTN+80vU7AddOl72YrmNykQP+JIFO2jW9tMa1RNRlfovgAjRiYUxCrQBY+F7QkM4Vg+TxKxJlqGYOIwogEUTgTjOQsWTNit4e9jREm6hjx4uyZVNTTlM4YsVzDHJH3lHNigusFacA61hbRwP7Jh2cn7wm4dzD1GxScTtwI/DEHacNCQiYNR/pqeBMBI2BvfD4NWmnCmOQrAzgResRjgT9hwF0Gqvfy1c/HJz8cn5IprF5ISRASLr4F3IVNS+9C6ASAlk6u5j5grgVjseSSZHsL1tZCpVybxItS0B4kJa2mb+pmF82mDw7SrtoN/hjg6eiSGACTSOwo/dcrbeDHYqcc53Z0yGKWjINZ+1S0izAmw11Ng15uWamWonYc4BlRRXcjYTPojFr2MH48+G5cfD2BwQ+Tx3blNc72cPq4Y1V6w9YfJt5j24b3ct07HDAYxrnZ6wzsi7FAs1KvzgGAoJpd7B8M75LzTg+HisFXNIllnKEBaswkAdVxKXoiubrAk8wQsoEFgic7j51PF+P6Zks0Ly7GSvbDtoiM/aE69lYlvxkvuNURw4tC3p5IdLTzjkrYW/0kn+VyhWwvFAKvUuqpLkUhI7WD7LhwJqxAOW+wH3TD0jcCkg9nKbilPZ5fnRaW0TqbEHj00E2p7xk3ik2Ddk8EloFpbgcW/OHzRI0ewuAEltwdb+GtTJy+JwdBDDW4GGS2cVRfJx1zXCUIU6oeWySBYOggSS6VarMHE7MnQhXHNatQnk6cZzSpiI0tCh0908eaaTdhoEykWbuMHJKMS0TyOLBZjiTA54GyTruvKxWwxjv2sQkL3xKEadFUA9VSSymwuKOrBSHV8c79qLAmV6N0zF2n1EgjYxZEExpkptTVhb2NN1Q+Edlj36ieGGUVRi2yUULVhm2RiLKGIHFhkc/Ozt6eOKcv3p0cO2+PP56BxhVfFZRwQhMB10kbraOlyCM8klEL0BqPpumSXosjeearwDFtzONC9JfM2zu4LLxaIrldSVMHfrWn338/aOPPuCrLlh9Oq5ZoCWn1xtmO3zz7OBJc6rT7HeQSPIdFLr376awA98BpiHM4JzSLkTJuaGgzrVXoQ6xfNgRCwEjJaAIBN4FzAQxRrehQFHYMQ4ibHni0IHIXQBLYnfgsMcgxbBxxWooHLdRkBASCL4gh4Wx4Dqd/AR/lugBwRRMEHnQSMCV8ARRFSgPOLJQXp2thDqsFmGAJuCxlYqmITwRci1E2EwZwgKT+knkK3KnMC5hlzIRP3EZ7OdKTAO5LCVXAWz3Bez90GGuf+8gtA05VAM3wBFJ7Vi8jFcAoUl0GXQxznfbQ3KEuX4tkyD8PWGqJo+YqJCN3MLQwAsFzCDvZuYMSsBneO7CJvwrYxL8dsOmY4EZR4mZ7uEfif2B0U9d3IPuCatwr/pEMs0yhYKbVlxCnlmF/WJwTfz3Oif8f5/x2OKdjWaaI4JY12Bjoe+fUdE6Oz5S2QShWsV80nL5+9X705cgm/j+DbN5LZPO51Ooy8kql7u02lVJlE6/XnnYNo98Zumb/8ylVNXpnOlW1C4DaxZpREx6WEFu8mhAXCz7k/dlH5/RNvzsi34BTBD/jh26w8hi6yAfaoSTr8GW/i3nZwwZsXg58/uO7F69FjRsGWV3kypEs6MZC08n//P0fMrE0Y9GSYfzHJI2AHCFrCTCXpTd+OH7zs5ERxqRMRhdrNE0k+320SirRR9SFEbTlQYjwIEoNcgZTwLayPBHHHJhMV/FIUsNFcARKG8sK6E2E6boAM4oMCxicrCEagmOeiv4TASBnIr2HSUZ/Nk8lNeHjZPYPUJncBCqIc/LuwykYxsufXpy9evfWef5fZ8en+c56IhhMQ5FvdTx/rYUjWX7wxFOUT7CvyoqEBvRxRHIWXBlm/eXsL8vQbsmWkWL1VlZ7JNiVp65F5kuFHNiWJJdX26f+Nab5cNet7Yw2OWHpKgkReF6+BaqXZEkXwGGK2TzBN0nubyyJjtBApwHIOXeGNIOoog5EgOacoW4gLA0BxXsskcn7yQqrjcBWYNQGO8q9OkKoWtnPbnj3LsYVP4Y/sb4kPAiIsuKVnzwh7axVOS/YFsEtqdB/q7x+OV2VDzDAybgLWMxyFWh24081LeDmtZ36AKHgVib8VekCdBnFGHkrAM5CGqDHUbCGl8AeLyglO9VtCPKfqgXLqZKW1N2ruQ9ULh1J7pIEPtoDWqYwgTQBsWLymwP8kGTz3DlNwSLRT6her8VUp3DOkuDWtDqI1cyOgmr+Mg5KnXJvClxHP2B51fIdixM/TIMQ/My5dDQXxGq9JCJB2hK+I9Nh8BBvCAvx7OUJJ7SpfjX3kVNhoZUdfKomUUvz3UKr3YiIBktaKcol4PzUiaU7FNGx35NgrJ4lKJ3LPdDuUhxtGR5P0TfWFDYAKm5IYelCbOuaaMI50SmWIUCesgihyvay4rFdkWgYG1In7NPKR7CTF+0fcVUWEfNPWF4ZEQdjopULIVGYRfmCiwX1YXSJBYedTeSybF2X27WCAi6GHRVp+VkBhmiXYs1OoeJ/2RgjJ9CX8OohWHnKNCqvTNwykkWHjK8QC1Is3kRTFVPQ7e2rPuDPf5xTrAtqbuDH8c1olEaRs6ThjQN7WmElkjcuZE80ssy91d99+YazYKp0rN8V90A6Q1s3rf2GVzjqjCqXUMTpZoSxoPASOTaq3Hep8a5qRLW+/0Mgq/nVS3HaEpTuk07wsZCPtXxEDbyggRV5ITOdrNUTInHRHnMqc08OjLl8zmNFcK5ecBdIls1Ty8r7emZnlbp+TUG+UblWgqSBBbDaMjsqaxRLw6Mfh57nletigguYdPkWf3Py31xnHfmeXtN7sae3UIXBEA9/Npy+rPZ+TchpAsfutIY5v1N37u7p3tzqjjz9AvIX483dC3FDQBq8hlZhTJ2q4eiVUqZOTFBPzYQzC5aM8Q84r36zkVajcomogC5AkF8EOxpGtHCixAHczbRffy1CCvxRtjEaHYczP2QaJilp+kDbviFyqOBTTaiSoC2HLr+UV3B7WCbWKOzqFu+ANOtYWB6zi6HlXjXcrXQosLoyQYGDdQoh5VFoUTdHbzeJE0C8Pp2FEU/BlWP+s8XjwN9cW5vW3+jIcsmauAUs7Mc25anItge/sysVrdmV4T+kexWk7uJJGwWbzSfgrnLTqaCnLj/hzqset9QdWFY7rpyQzej8ni7Z7rQB+jXtnq1b5r+zS96QT+/Lg2+qAv8eDnzjeuRnAb+t+y5w8w/mvUVcESmfEbnEK7ofyFNyDaeeyw+XeNRoLelfAU9fnoObkt7GD/F5calu/bdFPb9ptrt93RRlFTjeElhTyqvOMYpZiCrw/uyjscTEBcj90S+PGoYbrcJUq3oLN4g42+p/W9sfIg9LUod9eqCJWXQ1WieHq3BCA7w17ZFJQl0m7k4DycOiU5TjH2g4FRzbk5Q7V3461w6Pjg5BwQ8xg7VcYVILGwm2iRoBCg/zNtESDxGHRceZkSyLpn6Ccp8vna0g2i1WyK24UZhi8kw7NNaYxAH17xvtw8YX9C9e7xQDmp8ZcIc7lXX0hOyKkj7cV/A7zJVAJ1bd4h6UV1e6U1S/nboRcXEE2T/i0V/aj1BV3v70I7mKVgEYQQJuAcvN7uqNSPn8GFHvJU1pKb+BaT5Rr07YX5mbYtbuhoRR2Hp2+uLVK5ErlRpAyRTGBuTwp5Bdx9AVuJz3K9FzQQQU2pNDMmFTdHS+8AdoTlRdfBPn6EQlEjEHRTEDQG8IW7Y8yufKulVeqY332ro7rft241qa8riNXS7kH9MwA4BO7oUz5y+P/Q5M7C64g6d9BxkKuqQ1il6/oCM7Aohl28D902jJNNNpm9YuFdsxvAsj1HDLaQ/sOw63nfawl8/fc2yrfUcKbRiNic67jQLbGI3ePPtYGSySmJmPP1XZHZD1KgH9maFKZtl/tO8uCDTkUdISMVElO1ElQGv+/OpMpJJyalOakCm7Aje0pO4coi/X5VckKf41y9VI6K244rlEqSccCCcsT8dI/dhVrgHDCVmyVanJX6siTXvgDm1rahjt4WTQaU92Fmk2A7fqM5smVPgBRFtM6ODT6uSY8YcYLRgOLDWfPMRzgPEPtNShAJBPz5xnL/S6bso0X8jk2eRG1leUAY5U6Vnc7t2qNDfNneTkCRv4X7jurNEA6dzk6UFxq7VRSDVyxkKj/nsIrIWLXB9EH1zTnBbX8wSPLM0wuyErE4OJcDE7yYlrQZgrLVcFW7KYsIFgpbTdTmrqdrI6YBr1jKn7dK32e5EMwcEZaurAAUpgtsXaQP45Cy0AaNuovF1nb2OJ7fTdlBW2y79kqf20RR4CP5cD2/H1Sr7+3c1iY7uba3f8hd3Xn+2e86j5hTza3bHKvD2TZlzdT2xzMWn/nNn1pN08kQIiO7f4+SXn38U16r9Q3Pm5kTyQYPGyzpWKOH0EgR/sq+RMKw3KnXp2r9efdA3DG1i0123XutPq0JJDrTaiSx12+ggh8JF9kVa1TkdhPiERUW8CKKC+axQ4oE4i25yqCqL+m1EEIoCcSqPV0chYhVcJjfOTxfbXW4Ue/wtn9QMiVT8AAA=="""
WAVE11_PATCH_SHA256 = "0e644ae9c37ecdf73fd354a8c59100ed31bb3edce5c76e13b5f08ef4dd3292ca"
WAVE11_PATCH_GZIP_B64 = """..."""
WAVE12_PATCH_SHA256 = "9920afe7f160cebcb51ae0d0dfeca41ebfaa8532285e1efcb92245cd8fd2fee0"
WAVE12_PATCH_GZIP_B64 = """..."""
WAVE13_PATCH_SHA256 = "a94945d37f379119ddade5ee46380df8bdc31601995cae7bc02f31c253a2e415"
WAVE13_PATCH_GZIP_B64 = """H4sIAAAAAAACCr0923LbRpbv+oqOp8YhLRAiKUqiqJFnZEfjycZOHEtJasrlAUGgSWIFAhQAkuIkqdqP2C/cL9lz6QYaF1Jykl1VYpFA46D79LlfWn4wnYpOZxZkwj2ahd7Kd4/kg7tYhjI9msjIm9tJKiY7bx0EkS8fxHmv13e757bdPfXOjv2p6HW7p4PBQafT2QP34PDwcB/sv/1NdHrHp0Or1xOH/GEg4OI0Egs3iFpt0XkpPsh0FWZ/abUt8Sp++Iu/jUSa+aORTJI4GY2u8dfLl+LnA1H9mUvXd9IsCXxpidRzQ/gVZfGdcFOxOu5b8JIHJ5X3+fc0i5dWGU67+PrrxUFHfz46Ej+5yULItUy2Yu0mgRtlovUfX9+KQ+G53lymMN9luEpFNpcikW4o7mQSydA+OPwDgIjbuSwBkg/LMPBglxOZAeqkL2SUJVtYcJzR036QLt0MYCYj4WZZ5CTxJnWWSTyRIkhLsFzhxcutiKfi9h9f36hXAnZiuL5YwjSjmUB6msGLUvgt3nx/dSZCdwXbWgK0iVehL+AFxXPZJoapTKcygfkp0KldIHkaJ7QNIojExx7tCvzX/VTdX5p4C0e2/3ph7FGxRXc2LdOXXuxLWm2rmFz9phPKmettW+XXPEfKtcS9Je48S6zh/3iVAVIdpK3UYhLzg0VOQstYXQXcysS5W1fo6XNpsnga18kMc35snSG/nJ/C79/PLrBRr5M4TTtAHN4dEIvLFMO0QaTgxWnGRJi6C4lzq1F1Di6U8Li4FF8DbQA9j0ZRvGm1L8pb7OD+dm07yCSIn5+LXXt05z5n9/5PdvD37aLeSb9BLqeJdwQLA8YI4uhoEfsl0dx09yCSGzENQingu9RCmUV2l39s+6Q/nHhdj2T1kS/XR9EqDMuiuRE2ElvX6gKlWce9UyC0g8Ojoy/ElR6HiwuALOJNJCaudycjX4CImds0jgf/5K5hWn2xkG66SkAqbeZboJ0gFb5MZbKWKYiPSbyKfDfZ2uJ9ImE1ocjnIuZx6KcMy4VxmejZZ39G0YQEiIsOvwQQSZDNFzILPOHCJHjgcZfGLRXIjQv/ZMFCjmDSDDBxYQ0A8bgr3ry7en2UwuwQlwj673//Vry5fvcOpFcKu4zyTg0/6Z32bXEFXOXOJHELQ5u6CbxvqjgD5KKapIxmII9BxsJKgaCA+hdu5IFAjpFpWUAD5BjuuVmcmNi7iRnaKookiEV3izOEKcBkNsBaKcnVVQZoxWUTWgElK1gCsAbMOgX0bZSuQHjZPJEw5e0S7qSgLzJmdpjtMo7SYBLApe3ImMAL8XH849t8x18DDsefcCmphEUAvlOAHsPOa5Yh9glwbKoYwxZfKwQJXgGsd46agHcpA4TD0I2bLOEXCv+5i3SykIsYKKKYxvW3+TTeA5HxNEgiZYA3OQNV583jVEakOcapDKWXwSjEDEgg0HtSTwM0DvAr3gBluUpQWaJug8WhMLqD0Yxm18tWcHGLO283IwTkIs9ko6UmghFzF7YjFguQxMESAOC7FvEakKEYQk/FIw3JJJBuYJOfAQl6c+CjhAgoZ4RnQIieu0qlWjPSXiKXcQK7r4H9WyaxAEJO6X30bbLNZHrBpOGFgJ6UtxzYSpqUljMePMhiFZgKySgMJkiXEhaBs1xFwTQAfCE6bfHOjbbifiWTAPmYzQHFq2y/ELwYqB8HbQ2Lge/SXuOCYFk5RVlisiJUbnEKDK6wGOCRJa58BpzM61ysACLODZQVMJcnCfUM2lXqKScStY3fRbAeLRnCAMUQUI3JQFnsu9sLjYs0w4HAh4XEU5Lu4BD3BMRonMjR6E1I+vVCX/fwraORn8ArQPm+Bll7UboznQZw+QeQyoEnl1lSvqvMo9HoG/pwIzOCfAQa+7tI5gsopCUSn8VLKaQjqNhFSgs/4mev0d5kJsR9JH4mtCI6Dc7+imYlXN8HCYGkgyhduvDRZ0BuGEczJR7JPiQMonxBCgVeh012l7gdk1UAdA4Tc2E6qZI7cxelgXjz/geY3Z8++hLR1PpKTlYzS7wG2MClr8H8sMR7YIfADa/v258ODperCe7nystERTaBIXHIFs2R+J4oDm2DfGu9+Sq6U/Y3AomcjGTXiHV0/ug3P/JzbggWjr9lYkU7FshLmqDEBxAV42zMO+DD0uMCyhgkoTNxYTcPwR46FL2x0ggLSzCTKwG2QWYUyNxgTnkxmO0geIxpajjVafIKSeyWFkVXamuS26O1G66kekBc34Nsw60ZqyfGMBFpUA5OD9kKjPsS+Lt18xuuQ7mQqJRAkdFLjKe0kVV95qsAbUSgsYnMNhLfDu4BQddEDHtB6CeqluoVVchsiVWB38TTDOwwbZ8x4aIOjFezOch0AJ8gIUYgTnJFipMvOWpEbQhhJKYE/ldkwQA82cepj1eCogh5DeUgLIdQI3qPYpvYVcN7BUoTBDg40MyFhVkADF64fA9L5FSYz4hGKPkB4nONTDrZFgCRiollXZBxMxmRkEfqmwYzFqks80KW4DBfE2FdAzngfZjWcus5qN4puSKwHRon+NPCG7aiNnEk1FdNTzbsVavXbqvf/Bwhu8qXuLTQhSneax5nztJ8CZDJptBM5oYpG1LyATirgJei0O5MVqhbgMqWLvjPW+WNv7690iTIji0jBc3R8toJiKOf3r16Wq0hEtTqWQI1LPZmtcBVjVn4gH8TgfxY0wbodcLW5xhIm6jmFocqKaPkS4ZBhlmo7B+wAMl22Ro6BC5814r+1W+zNVxAY8SQUewmaLKrx2F0e0S7UBaFTRLwmx8LeDhrpTVQ32YuMcE4emE8FrWiw177qD8uI50Xg6RD3qGB9NOBiXT0RSPwReEy6NokXrRKaAevNB8agW1XGabn0caJwO1WhItoA+n28w37VSvkn4jaDDRirAullUtftQmpVPAHbXiaXAUCarWQvpWbeoBfD27MJG8hKAawFzwyn9BWJZJkeDAsmUnmeorGoHkiwhg5jeTOEQgeELZgU5OIS2EzI0U9ynAnkRJvGF7l8VykizcrFDAgzTZJjGEg9GrOHgRFFzTDoNEJoiRcLcCtmTJAhh5PQxBUT9b2lig0voyAISpOgClxb8ntUoKQrOyBkn8jWgIyNC6jheuwCsZpK8r6gNRYBse2ei8HQ5Z3FXFsymhzU8l3kZBRFBXw1BQ0vQcJbQl6P4hDxLREOeuRDUtvmgGPLdXk3ty7ZyXVswcRN3NwCkSEcRoCRUozS9gJCeNZWmYlHGgw0PMvU6RYDy0sk5MWGDskoWVexZ/KVEYjxKS4fCmeIW8+sx4ZjUuj0TP4YI7+tUEo/gOEzAKdDgxxLEHOoEtQ0HZAjAL4JPKFFUh3kRIWEP3INGrtsG5UO/ohXr9FI0bieUWvNwmW34COQrjQRJQebD8VP7XHtd5sN+Isl0tgtKAjG6xJDrHLBLIGLxe2FRAlS3dlYSFNTuNVwv4jIA0MA9B+12+v3zmv/nl7fTMihFyKwUUh/4ANEM1lb0T7wIH21kCHlZyRd2j9AO3j0tgNSUtCETzkCA2zydYwaSxW5eDEhcEdWUMMjJQTOcdgTcmHDLVQkNniFS6DDKcXwAEBTOwFS7eReNuHzdwKCkYBHoBrDJG1YGdU+6tzVnj6qwTDwctUBIfgGVOE9wNGGVIAgl36j4m9r+TUBWTtkH8NHk8MW2Iw/jsVbei4nrcCtLsZu7Xj78XfxDf/ujV16P2ds3C9lLbRegKI9wDiRxPAct0IgO1eohoMD5u2El107qtPgGNijLfYxmFi1AqrxtI1qHdVqD+Sq2PC5cC1R+RE4adUfBzfAAOPRgrG+FMN7roK97tVBn6CArxJAtwLdJQrYRIdvtFvwCk49FDDS+JVpl/T7FtU95n8BS3RvkwpOm+DtYVhMZRMGFO30AgunqnFz0YjUwCWZjVFrd1qloUW7cOoKqZIRO6YsrbD8kD7ZVWW6TumOYZPKPMOHqBhFaOvOtqPMxyqH3qxS+DCHf3CCgSmLoBBsZ2SgvBID7wwoOfLeWGIxQpAU+5eluV8bXLKIn3SvHe9c88O4E/O9Iiqit7J+bnhXs65xnoahwAbMg4b76733iUuaH5BkyVwS/7CokFelSkZF7XbPyAzX6FF+2QKEzvfaQg38RNIAKVY5IMXrnwOJXMgkbKtFIJnsxx0k+F7spLiEK6yCJV9X15AIToeWYbaJb0MtSPlr+u6ffA+oJyfrDosbIaiq4lESPYweQKh63HWg3IMGGvgAKrNago9Ygq9t3TwUjzPo5d7LawdJm0wzbPEYCE6ZBU7MnInYDS02gUOnj8XJrvYQeoo6pAOiLOzhqGlsEVbXF6Ks+ZBLLYuxemgdN+Y11keAXDS1ZLD8sy+lfhAW83D2L1Gk09tFOUq9g5G87K+qx9WEW9XY4hY5T6I9HS+ATDuB+jSm8bZ+H7M8YPGkIMlxncOmULjo/FafcrDU6G7pegUXWZw6EzjU+hYp1LDxjQDBZp16khkuLulCBalNDlmMQZJMWZ4ifQkBe/HH7UAzTO7hrj8NCbTCxYeb1peGCyX29Eoi2MHXQkHHOcVRRW1pQUkrNCm0s6YIgWixei9kkpNtM13QEoa4Xw9nHHTdGu9+xZJxPplhbymWzt09sGhWRlQoSBLqKTFS01mqLmIHi4rrMzsq/UuDlOhsUvlDy0Nvt3rzORYG41qmfwZ+7qP+lP7YHA1gILyq54wTdYsAlHLKq54xRbTZhqf1R4aV9a1K1hZYAAzxFH1siZN47ra16aRhZBqhKMKESq3OOxdm45ik9rgspRS97FSgT58d9fCzW0rEfOnj9501sJMDjLNIvYpq5PqvcfYFYhBTHm9uDA16PcbGfXtk07XPnmlIoUUGee4JkiAIMLwTX8w6NA8sQZlsQS9gbKMBErhwN8DLAcn36pZn0YwvmoWVW5R5C9PA8F7q2ZRnnvpWtWnVBakN6jdKVIk/cq9IhFyOmi6pTMZtbsqBdG1e/2TPabRnz7iRnzKkVS2mR1QioBmRweBHfBV6EJ6vwKxDYis2OykLC5NVBvWJuZTksyR91+0Gu1zC/EJYrg/OMGQqfkokMKVQGstURF7z01U9KbIh6GLDb4nWMIROVNBZpdnl7kBzu7J27tnd/vd6v7aNi7KwHTzynEODSvnhXe7YH7Rx0YUvMalI/3nKWyO9jIW4uROqXBkAS7Bsusu1ecgoL7K3fT9xPXXabiGDYz+N2Kp/GwjDRVDNO7KFjkgahrMVonMK42KigpwgK09VRNFuUSROjGKm6jwSLgiRes22V1xRMEzyuUFapOEW0qzJb5ZhpRRUQSVegRpQ8BcF2wGFHi3m7ma+DE36chdcUgDEzMDDfmOTmg6G8BLr+8wlho4HP3Ey6rnOBqByfzcYHurWZfvEgcYkCDPC4UAcsK5MzzvwoceMsPpoCoLADH9AQt4lYlgIcC0bz/2EhIylug5/eGJc9I9d3r9biO9vEHvIdB5C51PyR0ZleHXUb50KaWf3x1x7L8Adt+Q4y0yAOm8IESOKwSZyh5gFVq6Y2vJl1BhLgcowKFHsjn4MLS5wBYOauIUeYTdws+T2jhARVUaN51LSBqtriqYGVlznwVmD9FQElMTDYLOvzzj2F1R5CcDig9v3O2zfcCUC6ygaYf4hdg1g7I8MkDcl0DcVwTXs7wCKqGaWJXHScAGTfxQpumzuhRThH/LFQ6KQPx4EUQUciUycaeoH2eKZMHQOLGPxbtXeRVV1x6ewfcSQJ3rIFbSQktZVTuY6YtWCT0vS0tFlj2rsVKVaIsYhaPCIGxnxBQtdTBO+v8leio6BWWEEUGp6Ry8XwROjK935a/r5i2siqM8lGWVQD+KQNPId9JVskav1nEdUjcORlvQdENrDTZnJ8M/agcoo3S4x2Ad1swAYysetQaaPBbRe9JWfbYwsn7780Q/VS5GtiHJjVliLcUXEpQ1J983Mfm3KaWwRICpR0wkcTzwWaOh8uvOMm/lejYWeZfvqWru4Vl36Pp92x50T/tD329swGl4ulbmXbmPRd4nYBif9cQh/j4+wY4CSkHkDrbZLKD6YX49EFyzT8VJFOn0dYkJO+KdIuRE7t0Y6+PHgt06XZbHoor6RnJorXd9kHPKXmqPxCSMvTvRmoNd0OYSTNqZOUo3LDrJxAbUQfG8UTGDoXz2qj9mn6gAhVOdMIAeNWpVdD4xyNICFNV54kQeRIsrAkkkw2bKhwCElzLf2FWhRCFoAMr7kUhs2wWoD/J+FSTKw9GhtLygKcbiVZVOAoM2Adgt4EkSnJxwiKMCVt6HwXqjbYtx2X/PA2shVoWkmd4ZQMksm2NdQsesA9NhnNaYJpbX3uCGtS/Ibgn+DVP3t5G7wNIAs0Qbax5LM6OuEOW0KOShuiK9OMVacwniDUylQff8lKunOEWf110prD0pYNcxIuX1ppG8q4Tz+8X3UkQvv1qNqRWtRMpHar673nu3HsXr1AQy1S126jGC8vXmwF/pMS12G0GWqiQ7lcDCtHQRt74+0CQx826178gIKOZPB1POQ+Qx/HLLz/Pnedz2z+IMY+3d2gBzgRysbxphxOqrt4u2N7ayiYyVZVBeXRvTB1gMADovh1LpUuLeAV5VYxCzVR5fxBdrl+8brjHNNdxY77pB0cfa1TwA2WlsYqIYZO1WHotsfKhojOrsa4yq31Uhyfocgdwah5t7UhnQNroxja6/5u3Q3WGdRzajuuLGTWjcgDrymxG/A+nNCN+D7N2IbkJyA4J3IrddeLg/zSU5WqoQDjQWsa70O2++vzJyShs3VakCbEXKdA8FVsQ0l6WaVUJqcKKK6QJSh948jlMsqKNUPBeyUl+B2ZFSQIxiga0IFE6MqcsFLQvQ5Vh9gSXOefMrdzxQEZR5NeP+IA0vKNK21LCaBNOsnJat5yKL7OwkjsNaerYQfPWwxGs3Kio3VfkQxp/qel3pSzA8Skr4rw1YvqGWDpWN5IAG6/nOTMZcETh1Pe7bQBVPDRtoShlFzwU0/cwFR7Cqu+emd2mlpL5DpU7U84vWERYM11DYmDZVxXgN+qYRu79Nqtet2OtaB7WqIi0Wu6R8mXwAbsWKNTBa4GX+Bq2+q6NXRhjxD7GID3+nRXz4h1nEaAgX4H6XRZwbwmaM9DdaxLkhbJZV/EaL2DSEDcz/Ros4N4RNrP0ei3h/bX+VYllovlYNjQntLjV7UuuLSRjcZ5YT5mikqGys2ydg57XPEgdmow6+HsSoWWLGtW6NthAWmIknW/Vij1Vf7/RWxv0uVxv0Rc3F5mvKtQYpeHbal7Z9fjaV55PhTtdaPVVzqdV1as4/t07FIfyLjfkHQreMcqvO+JP4n//6b52KwEAmhek6s8RdzrFB+Y4TnapKij9vI++IREQBLaVzNDQ47i/uwEZh8RHfEq2rrz50ut2e8Fco1bikB4htFgIdel+mIAC52AFzt/n2XxwIfY27YowLxNLGdx/42I2y3Z3svOYa6vPLCvvecXcoz3u2PZ0OphPX24n94sHaBhS3aA+GtAdDPhyBctF4AAL1fo/UqQSwkAPR2LkpSh2YJmp2NnSKvQ2d4pGGTppyd0Bz7p5Ygz7OGrDcomfayAMbGczmmUMmUmsDjuub5eonumaJaKTOLgndCTFIoSm5SkxrOlR2VDXDxdB8wgMnpWZYLMsGWCWAzHrKVFJmeRAacqpifBOkOsFGIgMsMp41loYkwUPe6KOumpW4E3iNamPhnAnpi1SiBTfKG/Z0go80tErhndnnpSZ/3fesyo1woWYfmJjAgE3gZ/MO2SNqwkG0DtKASo5ic5DZsGKsGWv7AiQWRiLVyY9vbp3rt7djsZy7qUyLAiiwS+MlKWndlxsAWhOwjSzVQwvTwEZY1nh3QDNk3WVymRtjC9Sa3IoTbi2utaHm3VJlYUNVIee3uHGFuBWUVsJ1y7jW2FLnBqCpPclL5KM4y3eDSu+p2ZvEhUGTxr47RoSd9UFk1k1zKCP/eu9ULswD35elJ6bAMv6ILD1dvGRUPGLEe3rcxwzWLzgLLkL/hWaEZV+DC62URM8W36t1V+xWTk3kzTZxB+Tif4IDA/fs4jWLvNb7kl7Zwjawe7NWOZjybE2LFF7cxy1LAx97bF2fz2f48O7m2zhZ0Ge9G5a2+bA7ldntQW1K+XwfPvJGzdA4FChCiMxTZnUAT/pQzRpTsDhzDI0cikqlNQA/tsVN8PaH1ow294VYLSuTxA1GiptgG+rcDdfkYXny0Tfy3uYh8Vo5I6PqVgc0qeiBTpfBgyqoYxd2FxBo8UJbtG613LaVz/ApK8c6hHwPC0xUrtYxkwbhCktJmW204AS3B32nDBP/LsgqL6bTT9I0n1NaLljZix9zEhWU5bQ8qJCUSu7hQSWKmPEQFqAnZX6y+AkiuKOmoaZQxwxXdXEBRF5ITYmQDktILV8xqTXisy5mIR3jJTAIlhZvnay8O5kdqAZPJT7FeJGSBT7+Fmh9TFznKuGrjwNB9ewLVPqTVGL+lItiqImGoaG9DLSB8t0mrXnWH+BBSGenPes0z1qAauQGiZ/LRxIxWrGXwOaP3ABQGqNDkDhIfa6N0KkyNUh/NaJRujKniFWKVtH1vLPhWZ0VVH2hjB/oVUm8lE6abUFLXV6KD/DtBr+MRt/CiMpDySJ15JKnqD5XRhjhIy7znNp3a7OasFXMB1E9HPbRQDkHq6o33ItrRa1+Ah4J7qsXLzE3vsxLvZR0E3jIl0lXbbsMhMpQdSdw7rYC+bboCsh5nuLpgIy2yvuNQh7XOLmKz7nAowTItzLS5LrCGQ2VGrT8YBcYhUdZZJzi10sxND2q6CKwVJEBeV8I+jQqaWv4Xo/kb8u1XFF+ttRhU/SWUxKEqZ3j8vyDps49I4sMscECO8cXKYzq8Vo7xtZKWCvRzNJ1TD6XaQUFSqiOFCNi5lIiG9x7ypbXDvDi4mXctUv2H9UTH8NPSE1skx+fW308/qo7PLN6J49SfaWug04owJgKEmARgbnEPTmMymeAJFR4tBNcfnZBEXBoPLqgFDTYCa0c7LDrcfcnncFmhN8qsYNW85sb6qtrL7V2P7qcYuHNzttahJEpetcKLdFtP3X4+tHh8HJEh9WMK4OXcjLfObDGELtfqtj4EWB7sjFPy8o8lp0pTkh4dHEtHShr7+b3PGyTS8EdOCgd7Jjzfjvnz975CTnN/eOh1e8/iT1zA7TxhaeDhicu6ivIhUeqdWdJ3GildNjAe8bheUp3mP4z+eFYdJB7z/kBGY3QSONMZAhCxnMjVfOqPEbtLpPZRP1a2rBy00ZgVA6LsyIjX3mSG2wG993kTkd2tU03ovP1cPjwz43gzKP30I+NYu1i0oFs6pwyVcqm+t/ABLSb0c20QgVjj4qhxkqggtjMobq/xTKosd1u2DpCBxXafgRP/+r1J7Tk80kVZahMCDsBEAYaIZhFap8B5/rtLQHa6403M3KtxUD/aOG0c8C98+gQZW3vHXNno+fs3A8dpPuip69huImSRl2Ojjo6RZfiF/RQVim5rQsOlb1zs1928byJzzQjXKpom+q5XNgbfHspBIcXwSONjKhb+2IffCIcBT7SaAGoCxtcNwObdCmIjCsqPjgY8pmvJ4M+hzVrou5AlJpyhNmU06k05fw84Wnh8SNOPJ2mEgUNnvGwWmC94tChxGDSPzl1lu421ULw80E002UZtNCgVZCUrZrR6Cu5vgkDT9ZHUFnkaJRHQXOVcNLl2O/J6ZlSCSYahE5V0Bmc2PqIgS9Seyk1sfIBdalcunQaFmYqVOicg6MoGSlmVoCig3fUgchk+6MNNr7H09QoljkNqR8/z6ahDL2Jr9QUOOFdQFNnhaLgRH3SdBzN8RW6zHISx3diRsXCnAhLtYmomglWUaE8uG2hgKZ1z4zO/GLNQTlu4dLJBSlsagSztcXXU0DIFO7MixBn0RdCsJZL6XLVPtqymFubukGYqqSdflU+ZzpJxOUjRuiGfMhK/RIYC4jFaunD5HYUyZtUxSW2biKp4piKVUmcqOAnXcV1OvJhCbK+Ui5f6UmzHu9CYye29f7D9d+/fvvWeXV1+/ofJXiwrpNeP0e9yvupmDYeupq2K208aL9pZ2SHIMfONDE8P1X/DIYgVUWWrOSO2tocpAUU4wzPus7Jef+xsaqN4vi47wzPB06/O2woV19FHO/Eg8v9WBq+NQXvAiwJ0MdEprq4gGI5uJwSOLRIiSnwCYwC4gkyGBpXHRqEvjiI8qN04c1FfNasYP8crE3dEI/0elmg3SzPFia9CUVvpkZwmLUc8rNUp4ZuzmCmLnuchsw0wTTnzUhSHeFR6dm2lDmr3NC5s6E7OHX7tn026J7Ls35j7qz6aCl7Vr3JB4wP1AHj8HuoDhinTXdMn1Cdb0DeBzkYJLTQwiwQoKiMTp5tPZ/FwEPP8eRiS1y/v3HeXd2+++GtJZ7VgGNbB1dn0IEnWP5wQ9HC3oSzGRMusOhgxJyZNDWC+63xLHSq/uu4rQ4c+mOgWboFnQ8H1VmqcUMuvc0BTGICycduzulMNSVecsSR8rj+8frDPzFwYLHhzBUdRbNsFjO4Sn2GrmjQ1RmNNR8Ue+DCj7bORTM0VXVEjgLIpNQoRMF0Ep36oNbzJRZ7JQG+K18CeiK2Pp7+FHUvnk8PSlj/PYdaGt8kH4LRRD/EQFHqTsFn4yxuihYBH4/iJC72QiUg03F9a+Br2wWAWdJq256Llv9fVsOXWJGa32cPjWKH2vJAPrDnWey3fDr6ndxuZNG2vYo2ibukw+vZhmqKinTKJ80X332znM836+9801n3y6WDOwMJtcBBAQGm3BQX3VEcvAO0igaX/HaS+UzYRtoDmYL+ugRLZ+PkX6qsQS5KS39zIgfmokeJedrUON3BFn3VRWd2XqjzUPOyOIrcFsHZvNhcVb4gl9DxA8RlWDtEJ5u70YUSQspGw78YoRo6ZJobHur0MWX2iFnC6S639qcquLgNWY4O2PNCN8iPozI6cVi6lgpr9gR3jaCu8TcEyicD7IvnPh7HrcZv6yOeErEtVZE30czOUFJpXflVFCbSL19X9dYG65XOcKj2GtXOLSgDNc+q4CMtitU07JEOYKq/GfH8zkImRt5FlmVO3c1tgtqN2pX+qcpCTLuLqgkbprGr+bIQV1gVBHwZgZfbKoupkkcMCheIcS29Lz52p8f9C3HPAvDTxcH/AoQnuJMYaAAA"""
WAVE13B_PATCH_SHA256 = "51aed3b9a31adb8d902257c605b79181bc1973ef5ab123a5036148a0e6ca878c"
WAVE13B_PATCH_GZIP_B64 = """H4sIAAAAAAACCu19/XfbNrLo7/4r0NxTrxTRtL4ly032Jm22tzfbrySbfedk82hKhCxeUaREUpL9Ev/vb2YAkgAJynKabrt769NGEgEMgZnBfAEYeP58zs7Orv2UuefXwWzruef8xl2tA56cT3k4W9hxwqa1RSd+6PEb1h7ORj1vbtsd3r+Yuxes024P+/2Ts7OzA3BPWq3WIdj/+Z/srNMbDq0ha9HniMGjechWrh82muzsKXvFk22QftVoWux5dPOVdxuyJPUmEx7HUTyZvMCPp0/ZhxOm/i1tN01DJ472ibOOoylv6OX4d4qdstjGYsuZxXbwf7RNLRY6C+56icXww/H8FXMTtu11LbaO5FOAyWNnubOqQKlRksa+xy2WzNwAPsI0WuZAVu6Nk/BN/jtJo7V10irDkb1gjyvd0N/ZLH7eXRbfz8/Z3914xfiOx7ds58a+G6as8d/fvWEtNnNnC54AQtfBNmHpgrOYuwFb8jjkgc3eLLgkzGgEBAHCjMafhTB3J2X6eHwWeVyQKeDX7uy2RKfPQaOzX0IfnTIPpu4xZGz+WSEcjtdObmF+xFHo/z/eoFJJj4uxoMfFxWehxzyKmcP8kLVt2085iAHzLLqXSr8KpX45tX41ilWodncPAYl8/UGbyNcfdD4b+VB6IAXfdajz8F/7/WeUhR0z3ZyV730S7QBe4ofXAXdm7tqd+eltrQB8UPPPT6uhpNWw89l0kjrd5DCMs+5Ymv1KdPsMtPvc9KvQsETHWlp6h4yfvbvjnY7Z+snLpPnTHbmzoefZ9sWoc9Ee8sPmT9HabP8U5cRs3THZP93xZ2M1gW/9GfKSGb1aiUbMRlUaNg0KTHy9y0kUcMA38u/1xh2xJyz1V9xrSF79+BF6KwUijbs//HcY9xG81pseYDYqPAn5ns39gLMVaNyMyaQFLv5su9cedGfugHjv3OO783AbBIc4TYBGlLetNiDc6rZ7gPCT1vn5F2Am7uBFveeAu5jzEKbnhO0Xbsq8iCeAc3e2hGfs5/OX529BeKURi0LOvn3x/ffMnaVbNwhu2XR7+2eCJkB+47vXYZSk/gzqBrc2+ymOvO0s9aOQeX6ydtPZgvmATZisbnjNPTa9BTMUnvAbPtum7jTgl8xPBTR+4ycp9CQiSzVwb0HCMdEQgaxjf4YQOIhXeJDSM9f3UNzaSqfAqmUr7ibbGGrPAhdkDVROOcCesKvlFXNDj13t4BPAwFw8A+4A4NH/cOo42CrQAw6WswCHSAhcHAFg6WrG/aABjc6H/Sbwe/eKff3mGfSDx2zYPwNzgwP7Bu7UYty/XqSiNI1SsLqjUAB0Wb999vp79qZvs9cZ1mHEK7b30wW72lyxa38HL8M3dzqDLnVQdIFFc9YZ5+8UAPF9AMpdcbant+IQ8Bc4A+lixYE88kEUe9BR7Cygdg3YTX2U1hLj6T4SAIFui4Sttgkw+DXwCpsCAlGp4SdiD4ktiUS9uuYRvAb8D5waoOh0ciAEcFGgV1M+c7cJx6a3bAaMIxtwcFncVbQNgfwwQh5e+zB2oOA1ODUTAQfmPF+7sZtyxs4ApZ57OwFACDwjjyV/p4gE6CFoniXwgBhrkoFZ7hxidgQD1mCAaCb/CL90B0NC98uPbyXYrNkmb3dmooxF+JjDBJUse9I6aeFYSZihXJxMvgsBQJheZkViDk8m0y0iYTJ5jv0Nvef081Kv48XAE1DnA/503J3rBzh7LPY1/L4rVRYOXjKZvKQvr7l45wy4O2V/f/bq+7/9NGHbBJQn8PDgMiv57s2LV6+Lgm77Egd/zn7e87BrD87a9uD5hHX6bLNFZ1MITuSHLnv5Vv4E8g37FvANCHii74qNL4Z29oqfnW+++754BRTlb3/5Vi+DWVb07AdTO+wbzva1H4ZAZ5jDq3UqPWGUh2wBknwKsq6QB1GYd+WHNz++VAbb7xOOQDeREvvqL08bhEx2+jVpsxVMF0HqCfuLqrnmOF7APEilyeTbQCisk9Z+AXwtlM1foEX4/TatKLxyK1BOH0QT1YQUBGMfCs0l+kGGT0tYSK1ay6iV6WngvTiFkUounEzCaN9oXlbfR2zwi17347JBb7N54K4TsAiatpuA8zVLHMAWoOEx6/AhOxcch/oWHjdPWndIAaTqj5nmiYCY7Ar191Ums0iZE3gUnKC/sLh9hZyHU5DmKFDaX7nXHIgNliloJPY6AO2RDQobSNJb+ZOkeHInOeETrBSC5s/ZF/o8BSgKQmOebuOQQavGo1wte7kyTaDCZuuDenLZ13/75hkD3Q+9f2SjVm40mzoZkLb4MiDt1zT7hRuREwM6Q5wMreeRnayclfs/UWwx/ZkfRnGTffGENUYWG9T1Fthk5aZfNHQjqhjDm74yjGwUQK2VMxoAu0cpfv1w9+HuUckOO66DRZtm8bUeK1IMAmJyOTiZBJErLVTCUFGbtDQ5YtCA5BRrgWR7LAWTMpPCKU4jEknAw72uWgQmgLN2PaiA8sX2/J1DNsMYmX4sa0oOz1hVaG3JsexVtD8jDLDX0TPQkuEMdF4I/yekrK6JVy0w3XwyrnKIyPz7RRRQkC+JwsKkA0PJzewpj8/8BO0zlNshEMSVaJJqyy7GsoEZ8ZbPvtqOn8JwGiAZFBQ9lghQCbFy142P/kfWaPj2PnbXIJevndU2aHQvmsUD1/MaPuDtYtRssi9B4w6ExT1WQc2iIAB7LJdPML4XN2CGsnlnyLr/92wE/bnpgpUMEzLw0zTgZ6A5fTeEX0suQ50pTF9AL6Ixkwa5MET/4fD4wmnTngdu6tCwnI/sXfum3bYYvvd9s9JDhHojAM573Qxizg8PwldvJDDTIcwAOLA5Ot2B3W4C3jrdEXypwVXRl+ktcgz0YpPYAQcxVrRoyeFXn9+IR9Dbvvq4MoqaQpgYpZY9eIATAT405GqV+qgPnHZ/7AxGQ8EMw76CVlS9YCPBYDQLCTQY32fOJg1X03f7DQ4f2tngukSzHA8SfvPPtrdOY7VBMtMaqEgyNiIBtQBTtAHvstjppuiAUpSA9X0qYGn9u5mrb3OAyg2J/pq3iCpzgHajAyoNtFFlutox3+hjpldoBNWbSIlqb7ZgQYCudDbjjADYsRtEww0OuK4Pva4idWFSv85selzDWHK+Tg7a8FYuM9GhYHtwcDha4zm8vQ/eNPCYzZ7heFyyEaYR+FUwWrEEMgMrETwjlJAgKD3heiXYDpxT/FTFBF8Det4VzKpjSnI2KQuJKOueukKZ/ILK73WTjpDxxNy8mHEFFXNMPcucN9AO6LaA88aTWexPhZfekEoJPZxoPk94aknFA9UAqy76yk0FVRLaE/ZRta40w4qBxwZc+PEeK1i1PTKGu+YrsAFWLjCcU1gYTogObMkaEfyoP0NZ0GIN7FllUpSrwowoqpJE69ZURXYvPQHe15/cln7Xh6KkQWEsI4IaS7ROSVl+pymCzG12EjSBE52fNbNY6SMQsG3o+UTwulJyZx0FrNysACh4/BMhAqUq7Y8Arc6h1MnjCqb4ZUsN6jd8EEBNdJNKSKVVNfByeLhdcXyu2/uKE5XY2H0AY4vZBIDe+e9zea5YsNKRAnUgqaoJ/dTZOBSGqXZavgleIrBO7wCTpdR+eU972VgikYB0qkCyoMhB3NX1SB21/tbC7s5Ffh0aDnSheG8hCgt4uiYi8zLkCfBNEfIT/ieGesTEpZhYzNcU5uSl8BNV1sxxCrqJl+7cYMuJieibDQ6AjKEp8jQ30kUYDSNzOTw3vGUrP1lRTHXF3TAp3F1UclmEFR8WgT+7cO8/lXmPY9zM7TqIcc06qg1UoLknWzioUHZ89sW7Nui1y6op+V6F6KXRgvTfqQai+v7sLZxcCuCbeMtLRXM/TsD0zxAO7hxwxeVnwGY2QL6uGZxA8HtlapgGh82N0gM7l8pIDkEsySEsn2WLk/Sqco18YQe6R+95l+bdwk0latfU+ugV50inNorJ3xIA2nUA/Dlzwf9ypn6aNCgQMVV+fjAvPWb0m7tBwi/NdXRCgiGROlHs+GHC47SBVATDBgxWdCHyGIK++tiq+XmncT5aYHMQB0psQY4LcfM6WvFGgzhEfyP2Xme1T4u8qM4+uvWEmglLhAKljzsy5j6kd2ACB+zD7G7CPriTP9+xHbDAFL48OjK+ksukfO0jk5ahsrpRhCiU0MTidg3GOE/8RDEdZ6mbSMMxsxU/ktgswie43PK4FFKBZ7JP6xj6GIQqdh69kytiZ2guvmcfPvzj0R6chu36H48mH0RI9c76xyNamMdHFIrEJ+i54AN8Gfw+af3jUT7Ft1S1MBkmdg+bCFWcl4qfsmyply2VMgSdaa+ignyQga5U2Og1EAgi0NlgMTahX0v91077ucl/Y+viFcmac09gaGL3qff1RcRh+Ju+3N1pMT18jdDlzfJT6dSUHxcav1Kk+DFKkWK4nat2QH2VUo1mEa0mI+vOuLKbxLNzN015iAr6fBV52tKuqVQu4w664+msPbPti3F72pvNjTsJjO21FV5jDVpVH3etTp+16JPW1dfbKa6tJ5zCQdkiEDvNo58gd8BJhCdv//osA/k1PKEQ94sf8mc/uSnKIXbCMBj/ahuSSbSO+dwPApb3hqBRIBFMoihO5YTH5UMwYTzmpzZBODlDMLiwSbIA/pkttuHyT4lcSBJGwtXSof2KV+dXO/mNlmjlYjCPoT49FuCmboIrfle42yXhGWwuY+W4CcanPtIiM5Vk4WAijyWWgcHauhLwYg5CBZddr96FDi3kJqY9Yu+vbLFEcXg4AvoGmVbujSk66OP6C3RxytM956EAh1EIRWJeVd98RarbzSIiYtGQ3oNBjxja8hB7JeBRzBfk/HYVSiUQgYbNIwbFerd9CPEClob9T0e87FgJ+wXi86HdTwH2H+8w2rFvzAJ/vb6dTNIoclZgIjtufA12V5gmzfcn2YSQjCt3VqmreidqkEGbKaJkM2Ff/00MrIjAqFSdKDtFJBpLTahoV18EOCg9xsnd7dBW6W5HbgA1joRGA3iwy7th6GGOw0rlKC72ZTVUkaqOTDbLt/qAlMSJ3Txh9XJSIlI+stfpjS7HquVSVo5G08Ecntgjb+CNhrNaWWmAUJGWhjqE0vFgaHU6gFT6MkS0vvr+9Svnmx9/eDERg7RRW6yYvR322dq5tUxPp+WnvS48BcaUW+z056S7BIYNBZappKDBCe48kjuf7JhfQ0VcxP5y/VX36aX6eAoNv4y/6rSfFuslmWk4ka6rewsSxygUhPwQAiXfDZKJqRyeFFcCmi+mK/SagwHFcK0fY6lyDRbtTACaLmz2k5vgTkCBoRxW7kIL0SHnvfR+yZ4IbqWBqA4w32YIX+Ed9BnN5xoqcLXky/lX/RKCgHRfxp7AEJP8MKTteN3xqCcmWZkbAk8QhggPrbsWewf0f39ZLqbOdagUBZS5XLQmsufxJ71GPrx3KhNk4FbRTtbrwcDBMPI9+6ZSBmb3l6CcTUUDKJIlYvxi1393PO5ZF6bxz3Yp+mT2dRBN3SDDAr499jqXh+r0qU5X1ol5NsIRPh/iP51L3c1CBvNdqZzBGfDZl7h5REyp1TawkUklFG8gwWSLROASZCiUbFF9B3IdsCVCPmc5L65czw4iOxGNgZc03irYbSTfZOyJaNeXo3U9z04EGqgXEmGDjB4lCISSkbH52CpwOZJEu2h3hRC76LTR/IOHz56bZRgJE7Epv65EiC1DYcijm7I8I+kntX5JbqlFnyzTekaZ1h0aZBqRUsC7BOmRJGL30eN8dzFFQcqSpV6imCRIp2sWIV1FhFx0RiRCLrItvSVaGEVAbsiY5USPKiEBqhVEFzp9qiLx/UnCJFnEisDAjpUnJOF74Qa4Spmj9Zx1TVKlThgNFWEk8NXvWWPE12AgRe6PP7145Xz3zf+5X+7IySRfgn4J8ADYTjdkkbLWE5ZKK7Hgg0s0R2EAElXv0vd2uY/ddj6A28tC3ORCoduxhHCHr5c6ckqGqalx18pBdNuXJVkEfhqaqnJ9rRAwpeaKGBIwzM2N0qVDWEM4BvnSV+RLZ3hQoHcGolY/I+R4JMXQxViKIUHKWmMqlw4PFkXF6QVDKylUWgeLjSIJTMVPNLO6JjML7KZtPOO6VMK+f1ahdL9M6rXbHZRJvXZHm2P1UqlzjFQSogvHc1BsKUg/Qi4VJHhvNGTq5Mqg3sgZ6kYOYAGNO8BGJqEr2HiYtIEua/JGo/AR4oYm5AFx0yF7gDQ/TslWTQVFJuQztzL3heQy2xZiPstZ3amD0LVkl/sZOoddFOC9NtiOnS7i8+Xbv99jfugOZVY+F5NQ8VxL01NzU0/Omict4zRWfVaaxlX/Z3Os83OJ3jgLcQcxyyPU2szcKB7IBl58qdURqJUFBnEyNoqTfumxnOo9nOsC6eCg0Iy+6AoeLuPc7Kgs6+wHMVV3dcXCugBM1M3gjTKFNwbj4oHyRXaqXTJqDguZ7Fzbe4mjTrtvdXqApE4XvoxNWDJP9JGc6IVq3jFR41AraZorrZBzsnZiXkuhQfEyYMJzrJFLjZK9mp9DesKuAZXf+Cv7Jttdo8KxZdAM/e9g795q/vM7XLqwKsDfs2kQ5VFvdLSgIxi+PHJWsIa+ygRv+vnl22YBL4nwhI3ggQQ3DymhgSICUBmzr3Y+Bya0oqWeOakI0T66VGFJO6hCsk9GDVWLO8YKJPlktW6N+OtRe6zal6K6RoD2enXO2zgvr0pykhNWSar0uzWeplKbvmS+b02XMiGU66/ljvCP1jw7146GsksolFYkLgxLuxJ5UTZ6rJpaTPe4O4RDemmv6tNLAGKO9kYXpDz6OFdJeTx78+aHV86Pf3vzhwY5ToN0hkYVMuoaVQhpFoF6UB2oQvpdeczZiPk/9AggCHhzgJgaZHrEiKrDyuQY1aGoCaEb4kLIG7QBaI1RoRCObiH1BQju20SK2stM+Cuh3ePiwvVSvmKhdxTzvU4olNt0ValeaoN2thiiQZoPSFBj+9FluVltGyG36lXEgGTa4BgVMfiXVhHVkZPzgyQcHSgeljFX9RwGiucw7o5I+I+HbSn8v/352cgk9x8m181V0mj97yHwL8wuw8U9LsO4R2sbvfFIBgHKyP5D1ANu2sJluGiPpagvY+kPl+EPl+EPl+G3cRnu22Rg2oqll8nNBf32sDvGlC4wbpCf/N7NBXXbsErlKETISuyzFn3SlgJ/tQ6K86Zq3pKpafuHwMxqUso4QsvEk9J5D/O2k/L5ZOUYjbY9t4H7hm/FQfYp7jtt3OIO1MsiMVBeyxO1UqqFcXnRH3V7bKWu+IiTUhtL6XXzUu8RNqBZLw6l6PmOqLd45uUxfnPybzNnF/meZag9fVBt70G10wO1W5XacXIM8PflbHBy83wp0w0P5vYcT806uEAudtQL3huPrAtgvfEFLsUcZD1cTZywaRQFyvul3qzjStQMvwIH3ghGmWFSONrJ7ofEMvCcnuHvS0O7hai+kPwWSjhralxJPmfRkLWUWDDUMgNLWj1RxvVQFr15EBvBCA/UJ6JekOE2aMtQbz1Nc6CLh7Fy+LAur39lzkcsr12fyFBs9muoS87NuomCCBt0cfdKa9Ad3z8LQHOVBa2ipspFpglQrFh98gSQDJyo/OslqYPGmoUvUNnVMA+yCbCUE0JIXIXxcb1UHZeJ6xNEdzGYh3L9wzjuPp4f9Gh9Y9Bv/2o8v3xQ9UXyIKZPPonpQ2J4MKOKHQQH+VyI+dbgIlt5q0cT+c4TTBxgZmeloupKl5m9bqPrg9h9I9hUcutOfETEs5jMULL9Tn6CHX+Q69dy2izWOfvL6TQDkDoplTmRbT7QUyIaUheWzSGipIuHwHVUmSwhKBN92UgrqBSo0PD5YFto8yAOXt4z5YZtSsk3BFP86CmXPGzSzx6mGd2HKZrNp2maZOHGHCwpmRyjyL2pFpQI17SjJZ4ZA9OfN+hgq9Z9yf2TyQtKW1ac1CI8izjMsD++D8+/gznrz3MN/CUb4Rm8Np6i1Rw/eDjKHpJ+hgfgSpZwohxcO4yeAZo5reGwdx96/pAnv295MuwTn2fhxv8N8uR+n62cUxuzpEp84ba5NmuNOu17tfnRQqAoTaP1v6YyL9my988z4wswA402g5JUTiDKTlyeR4guk5G8IadERdxvO8lGHZpko+746En2WWeNAXz676a066KOmKY3qMQbs6cy0tjrD3tTd27bo0FnOJ6OaiONebtKjDEvEavRQ7Fqn51XwmNgMqHht+vtX/FcXjYyKGrM8Khrk+03Eyz+3s2OsallywNlu6JMxtZFgk3KY/GzxV7Ssb23MgUjHgZyRdrUq3cbMgZarPt4uRNz1w/Fab0CFGA49m8scVKRcsDueczZdo05+biXAZtFYepfb6NtoqVtw4OkOahipY4y11IeDiWHb57ZSSbChac7kWEDByDSaxTAKA+KH6oJJmm5O9KyUOJZV3G+KV+HQOAFmCLrBgHLk4jYetZhSjhbk3u4AJZiZmY6bqskEtbO11+yNLrm0I24SBxMSKVWo26BNnaFySuucrwzwe14Jsyj9FkCzyhPoiTxKa2sOImaZeTiascA12fywD8d9XUFMuaAoCnmYQFmwk7BCLC7c9cPtnFGRJXb8Mj2hP24xpF/JfjuqYEpo3qGnW7y9t/wHSULMkCYLk21KNjWIS+o3x2gMhbzK2slp9nfo3iZgEjh2TxDHDSuFr7ncWLwq6Y8Ab1NOOWilglosxRpyAspHZtNZws8eesmiZ33ka3nzs2EZf2y9IJQLckp8CNNuJ9evfjLd3/9q/P82Zuv/wttQX3+vb+ivND5nnNlGa6ApJ4CVqY3Mqe6vkdZff1UJMamwWEGXUxvUoCKyW0o0li7yhEd9XigqxwOzFkCB0vMYMbDpq5gWVeggSJK98kP6w/VsGqFtkqCRLTgNmNH5gAkDWOsg2S6pziU5S1zOQz8MIDN4eLl4eIMOMWQBnQ0YKDcv7QD60UcSmwsoiSdsNP/go/vUQtZuBSnmJiYl4OsxW3h8AHln2eMLdMDJDiBQDY0NA4VOgPTqjftEy1ljFZNan6RMb31BN3hpIE52GY2peZ/zHqX+FLEqyXmCH0gJwsEV5s28rkBP8TsaDYzKBu0J9UJgrOmrhMbR3aimzent6NFUtdGvlBttKRGu/qhFtJFa3cNgomabteCnCOyBwcX4EN0LiRBt2vMEuRE61Q6JTvMvQy0aTRFwtHdR1mnITNnbudgsjebdhq7YQIagDfokDcT5/X/RtpZEw9mNS1vmdr5fI8VUpFEX2RLKHJ1Z2i+EjL9KstXgzsK1sBlPt4AkII0AZv62asXZvX+JwFP1rNAz/oe49A5fGnokaBy5Q0kJQ0ujycnlH4eX834DQxcAHz76tn3NkgO9Nik7tsmMj0/5rk5K/ogx5nfbKCO8xVFP5JM7ZZFMLYsdHEq8+NzgQbo4s9jp01Ja0U+sXM1mZgfooVC4lNLp0+w/FgOzQ0R1lTodmF12ZTQGggPDN8ocmbliRCyHEYTYalriVZlKZqVQjrkBiJak+VHO+2R5l5KPc7Kn7oNoDuhRaq32I/QzdrDnNvDFNrvtFSYkZtreczB2ziVqX0tln1rPn1Kub78SE1uRNm6ykl5Vx+ZSMZ0urL35exTOLi/k1U2mSCtXkcu+8Aw9anQFuwOI2krWxjA7MkTwFz+46lMBVVUbzZLrpKDtZB11MyFh5Ij5wmmAAno58IHQ7/HkEoKc93obwOWgKGqsgARjBnRmn+27q26PL7qrqZqaaRFoikti2WGvwKXl+XU3ZRJI8vnj2TOUrHlJLWhCklVG6wXDX3I8zD3HEqlC/wymaAVU2TJyN8A8li83tRa0v9eCA05GkweThJeTcOvsgYaj6dIToWSops2iC2YoM48jlYi8xy0U4MYRX8MVSX4asp0b6OlEpbv0lMgV1IcexuRoJadivpanj0vqUA0JlauQk1UqEoCZS2H+vMzEKvXmQdDYl6VtZRFjL0go1UR3tNoG3ouPAtApRUp1Mk/ZD+AY0Y3o7DG+GIIvR89xiubwJnv9vHuCfzVzPNkmNRCsc2Ntsf5eU4LkQnNFZ/xNgxJK6HtfJuZ2qjkz7drJT3aHhwJZyqG+UReOxDuJpOdGztR0nj07V/xkgDn+es3z7598ahp+4mToDRoVjMFF2BAQqlwSxkKGynKAIrGgRHcBniug9n4qHJDklmjjFVMwsz119N7IiukOnelGzMPlPggzdkr3Rhg6vyVJsfBzJkrTbQ6JEdBEf08fv66jBkKoyYTHEdJhmXp5AGuQWjfVaTxD3nOal3K0c0CunQwZkMn790oZnoq1UXMNhH5KUQ4sPiO2RgpKtrGzMYs+5esfEUcrciBRd1b4hGAm+eYzyVrIR71jhd5982NqhKx2p6SVpPxRNZl5bIPS1hQIXNl6m+YjuS4gjm6D229I8p8UmdGyVaefqyk8eXi1iQcAOWzxIT8Y0NvKcpYz0cZL2Vual2KS+TSCZvaG8Gw4MXQ22VmapRU4BJ1a67mI1RPFEwXyROp0xkAnWQGYHeGZxnPHzsCKfVrR/FLB1HLQkeMR83qqSpRnGf2epsspOlaHuOeYlMVY/CXkFoqU0CRnLgHsZJNwl+FZl5OLSk3DnYlm9rHoLv0WxJ2Ulho5mTngsKTzBg01hHz2KohqMAputsZ5pRCMUoszMZyqW19bGNR3sWq8VSk469ySw2nCH2ysQoNkmjYyTFTmI+VUd+HFYGRSUnSWYX+UTWLj/l0ie0x7asjLGhVpeA6H8YLSFGldgjGJUjLbYgXmOD64XHP1NybEqIlfZgsO7e8D4opQQh0Z+W9BNlawjrmCY93lPk2TSixGDwIUxL/9omMiKAvYsj+Ryuxox7tqxpjYqVeHhtECmIQTKTBLC/6UUbEGg2MYTRbVJDXeVzqV/kGZNUr1Uw6Btx0AC/9deFGncp7UOBbYKOThh9L8bGr3Joqe5AJL22tSP2j1bQwinELuhoW0t6WV0LvrXKRLAYGNIev0tPM8as2XR5sujzUdHewaZ27mXUYw65tU9lSlHVMZTtRZlJVcjkDi3uGW21xGeNAb6Oit1X5upmoUT1D8+nG3HB5b8MlNhQbEuiA8Ljbs7od4yxQuVOslBFHCC418g2xuFJV41EZD9yBpTbdBq7un/kJhu3entHKXCNwkzQPL+PyQYuJXRVNS4NHUb+sXuYYets1aDpMAbyPtoGXLV1QrI+WKkimZkt1mrX5I0XPZgs/5Gfoo1FC04CupsQ3TTm0WLnxki6vmbmYnhMcULl0li+0SGC0VAbSNYZ6ynqIH855HAvxxdBDB8Cpv4IH4Lau/DSlSwtnXFz3qcATq6aWPHoYiWDiIkrpRSVjN5P8udiSwkHGRwBf+OVj8JEBK9KCRO5D4kVX27CINZVByvvSBEB9VtiU2FwNrtFT8LgdjFhq78vM7nLlauimVEEoFCeKG221i9yQElykBRdr7WfwUpEUXMdNliZb/4WDpO93lYvr9OZW+c6PQgvUtEPQSvDr0jTPxBQzig5lilkiDb64o1iDgnmvydp7IraR9Ns01cdDZS3MNNPJYMrWwCalO4bUVQphc5dFkFja1Fvpa0LZso6pZXh001apKcnhw22NS0KGXmzuAyRWhAwtl/e1lG81NN0d37Rut4oILVW2q+SP5X6V+bw/n7oz256OR73Z2K3dr1I0rGxYKYqIt0a4h7+FH73+QebCQ6VRTIvEwpfnqyn3PJSEQRStSZ4Cn5PgTmwqdIjbdd4m6XNKi537pLwXTFklJHsVANF6rQj9yF8h/aycg8sX9+RqnQJgowFYar92GTRVVhe7VapbSsoL96oaJOGKK0UaNLEET7KFQh2Yjje1cfdY6LHd4Q0mcmOHBg+U7di4vyS/nxIX9DdicwnCHXVF/m6kmp9EAdDI0yDSCPM7d7t2p3cjdkbkUVA3lbdUahpXA/ITj8/o+OnmfMnQfEjydPDXWzf2JuwKnpFh4YgLYvdusEyUdUkNHIESW3uw736oZEOnnRKkP7M7jEEJ013eYiuEESAgUuY3F3m2szU8G+yaKBEXuwcJLSMWC33qxhldneNVZgIx8j7OgLuxvOQn3UdnAZA9AHOEboks6ff87gNyPnEylLSkUT2Xl76Edg4CsyHATk/h4YbwjU9D4LLs6bL0tGwsYP9ozZK2UxZOqiaA5bPyTmjTPPTn2pAN7hOl6Hsi5ySOQ7nEMN/PLXapgloVX4QsLz+APmYKQokpVCK6BPBo8aBA0iXFN6XLAchc5bNt6u8knwmhgLeku8hHyg4eWuXQoGXbdFjqLrm8q0DdneXx2N+ptqeMlRZ7dKwyy4vrCfbIp7R7TN7OgEZ0thZOZvKUw4TnueQrMSzo32yb/DL/tsu29N5L4UbOUBYzfb2fUioXWjoDln7qtKroF/SM8x0oio7A5xrlsTzTDRUoys4OBQY+1WBs17UQcDkGPrJ1w0JXJRoEJQjbFGkwLoYdqw8z8WI8tgade+1B9a8Sc1D/ltolocJwLfbraN3F60JBDOQhaHFJqBHqXTUOgH90PSROjIa5PIv8WPXFywNlpyQ/MZJRX6d9oEzjtvpqAj0Hy/HGyUPlEqGHYRwoDg/3EunyWxJg+YkE0Cf4b06B5b8uBXb/HhTY/TIKGNdBxI4hiSctqpFHWJyPqlqrvddOiT8Je9Eix0G9KRJN1klp0+5BYJUNvaTDQfHTJcqJXd+YwvI0oCbuTFIYrbZJzmwY2d+LTZZtTVfXqgJh74XmxR2dDgcKcYEdO/vhcB+PHUsxee6ttjyijqJP7q3bPqLOpn457MAcO6beJjmuXjbnjoJ5RLXwuNHUzsXfCYGXn4nAJen5e6fw8n8PhXf/Oym8+ydR+M5cdFf1DnCZn52YNJ9w0ZJ14OM20jSiYBJ4/76btDbLM/zViqM1+J2YfOyM9CFDhYmnluimxbZMLt5p9++Na+rrURQio21+eN1xwrwo/FMK/27BRz6jlRW72nS9cBP+RSN1XOj36zfOsx+sOjdMvV6WknpJvpxuoEW9MWZr6aQK7ww071T6i7qHTA7axim5aK1jX3CEpXC4yhFqQ3b8cKUHaMrqkO+Frd028iCP+e6B5F1+EnmXBXlLIQ8abHZU5Tci8PJzEPghctQw6Pug/xNJvPskEu9+1yTe/f5JvPv8JDYuZkyYy96FFjMmXsUYq1wsqQUZPharFco5rCw0nEHL1Jc4Rt7ptHtHrPhqKpyugaunaJSIVA8nv8QI2Ojp/Y/G9tJGrS2Z8x7GFHgYjAQeOoPfIx6Wn4CHu+blAfsB3W6wH16+RftBoGB4IVCg5ZA4CgV5AvZ8JLV1q3c2/BNRIlgDIw1oyD2EPUYSN72LfwXc7D47u7gzYW5+XbDLRUegZNB9IEpmxfXKNS7Wp9p6m/uxR9sBljsbN205y0ZgsXbz2Oq7e6tnqz7G3RcpT9LkfI25IG61/Relguzi4377go+6tj3odEYXNRcfl5tqezDKhWKHDwk6/Oj0e/JgMDnKq5Xr4Eozd0i9ODMgvZwdyBr5qV89v8Z+EQV0s3eqHnWaLVwf92gkKR28mOH6934hMlL8fP7yXMYkKdGAemc5qKi4Go7EgJ26Hq8eqH0ORojFXkU/vRBntV++FVFLCn0W19HjdgrKwgA+Hi1t8tJNxufKNcbKYiotfYrzs/ri5+EFTwFPXfW02QtclA3xqji54grNsvfjxgBcgN3KVX1gt4CJe5Cx+wJcFGMW9XRPadXpUgEPRweuOdu5wRYjpGI7oMAUIoB+izmB2Cjubf95z8OuPThr24PnymYOGhnlQeErPP6G+dLPitcU+BT7yoAQ//EOuew9HSGWL3IKRhCbyxKH4t+4tc9Z53vicNdeuoij7fXCyQE3muq5XnEeVkaLm2gBX6+3UEWs22YHV9V96lqyU5n4V2Q9FdnsV+6NM0tvaBV02KcTTRbrZF+62WdbfpFV9F3uAnxmORYphpVsnVY5fafhwF2+4UHd6aBU03LrKW8RF5409T0RWQMpAGmjPo1USyFadAHjOVCpY+MppSKjLsCcg+1vJ5tY7OssmmzwSkbMhwx8sWsgNjMfGBCIN6UCLHWUS1P9LAFQB+89LTXYHW7QrTTYkAeSt8j70gMhbQ/0vmg1C6D9StVZVNSrohBTDmO7QaUziR8e0W6Yt8vPgX4XpjwOuCslEh3lF9NZRKToYAOSVCzrvNuwj2wJ/+/e475flEi2fkyDdiw9YTs+++JdG6h5ySQuJd+9V07wUYb+tm1TjdIhPnEl9mkG8l1agLDtRgps22lWoeIftHxn20SP9zbmblDPF58SIxE0qqHCEk2aJVCymr7lxgB4mQMWVVTIspERdA7Ttg1Qd8dDvVPnS36WsaGjv+hAKyOMacupWg0fqxNOLevLi0YzZlMvvRU1qtyollYAVt7/mPXV8py7JQ60QnHcDk9D15xOBTUPONFSSEwmId83TmU+AMRacRZIaY2OFMwLcI4pZ0Ov+5QOqNr2k7xHMnLRNGZB8MT9q8Vx4EZlJPnJ4Pz96t6wrBOSsMWwQWm7c9RHdP6a2GYyIRaK3T0ovDhNGlnvcd0XQAJkMK/SyQSTT+AxqLxcnFCmztxVz72vMQ1+3gkjnjwhwcQZgAytyrZwKNaqC8FVW13k3FeSAEwP1RYiWWuwPNhgmTcobqfZigMLhfkjLR8UWCJ1Gu0pE6m1yALA6gbbRGU8MK2gHx/xvAweZl4u5Zed/MTDdnKHV/atJuXJx7KgXM5UvqJt2dUZaeapXPf9chARHQYowVCERn3rcjRPUmljIbWr0X4VZv54k5hPX2iHQ2petAS7DNjEEJVUbYCmsgfvuJcpISA932QxPrpWAqdAbi7mb1FtInpARygsJuYgdc3oh2tLampqlk/pI6FG6yQwxS/tIeDvIV0shUuM3ZzJrtKbK/2q9lpaqXmFf2JfdzM6UPqbd7WSATjg1+7s1tjpEoeVXzwrPdjNqoeoS09K3K4XlrGilxLmqg3yFLwGWDoODfkxyr2rRfLnmXekUXHVFdzPEPy7RtNUK1MdeLTEaFBv5A10OlwvjRYkfU+zxiLdsekVWGpK9kG5dg5oWTBJdSV7WMeWq+8OVd+Vq68zdxV0qMKensqHnrpC5+205NJGTqpbqck7Ih7pPqgnfZt6awXKdVdbhD4qfceKVuk3GrcbmXSkb8gQkVdqlHb/19TP/PzKGGsLDIOnpS9Q6vOYczHgCiO5ScLjFPRqmjh8o4zyVA5ffYMgpvLkEcbzlNv4zmiUIoC3y+6FFb8fKf3Kz99T5r6Fu84OEXDKmox5dbdgiNxSTMsi202LOeE9YGx8MbQEFB+94BX3fDzU1R+jDU4FItbc79BdAJ1B/8LqZOkpKwKUgkxcyCEK4eEOSx7zcMYprMTkPGMK0vjmiwYIoMDWU0SjSY5HhXMiZcFxMR/wtEqxACuirZNJboFOJvKkcc6hqBvR3sBPVD6UCok8kpY4a6KxEEh96JIiww684X51cVpePPfKCsQQwKpZEfUqsMry3qvqmtpxlvsOg9ZkOKux8coERIJYJiS9+OFZ9uMnqDOZvAI+yQAcVAKC7UZiiWPYGedJUfc4WzqUrt/xE5x0MlyeRsBuKcwS7hXhTPiG0dDVGspdPyjYUNF8Zt2WP9XYUEHPQaKpCCwjzmB64GgacsxjsRA67A3+jcZ8kNb/H+/FVZoewQAA"""
WAVE15_PATCH_SHA256 = "9e2fa5aa60fb86a300e353ba7c54b268e4bed42f3c74e7f2e24a24be6bcd34c4"
WAVE15_PATCH_GZIP_B64 = """..."""

HF_REPO = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
HF_REVISION = "9217f5db79a29953eb74d5343926648285ec7e67"
HF_FILENAME = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
HF_EXPECTED_BYTES = 491400032
HF_EXPECTED_SHA256 = "74a4da8c9fdbcd15bd1f6d01d621410d31c6fc00986f5eb687824e7b93d7a9db"

# The user asked for two repeats. With three arms that cannot be a full Latin
# square, so the orders below balance POSITION SUM instead: every arm totals 2
# across the two repeats. gqa7 never sits at an extreme, which is stated in the
# report rather than hidden.
PRODUCTION_REPEATS = 4
# The number this run exists to replace. It compared llama.cpp's tuned path
# against glcuda with no optimisation flags at all - 1666 tok/s, when Wave 11
# in the same era was already 7397.
STALE_CLAIM_RATIO = 5.85
STALE_CLAIM_GLCUDA_TPS = 1666.0
# llama.cpp, pinned. Upstream ships no prebuilt CUDA Linux binaries, so
# this compiles: ~28 minutes at BUILD_JOBS=1, which is not a typo - the
# FlashAttention translation units take >8 GB per nvcc and -j2 OOMs.
LLAMA_REPO = "https://github.com/ggml-org/llama.cpp"
LLAMA_PIN = "cd26896"
CUDA_ARCH = "75-real"   # T4 is sm_75; "all" would build every arch.
BUILD_JOBS = 1
FAST_BUILD = False
# Raised from 3/3. Two of six sessions in the last run had slow LEADING
# iterations that never reached steady state inside the measured window,
# and that artifact produced a spurious tail veto.
COLD_ITERS = 5
WARMUP_ITERS = 5
MEASURE_ITERS = 10
# The repo's standing retention bar, and the band Wave 13A measured as noise on
# a true no-op (worst paired deviation 0.61% over four repeats). Neither is
# moved after seeing the numbers.
RETAIN_MEDIAN = 0.05
MEASURABLE_MEDIAN = 0.02
# What the isolated screening measured for qk4 against the ROW kernel, medians
# of three counterbalanced runs on this same T4. Production ships gqa7, so this
# number is the transfer check, not the retention question.
# What the first production A/B measured, at 2 repeats, before Qk4 became
# the default. This run is the same question asked properly.
FIRST_AB_QK4_VS_GQA7 = 0.0365
FIRST_AB_QK4_VS_ROWS = 0.0859

SCREEN_RUNS = 3
# Pre-registered in architecture/glcuda-research/wave15-qk-design-gate.md
# before the kernel existed. Attention is 34.6% of prefill and QK is 71% of
# attention, so +24% end to end needs attention 2.27x, needing QK 4.71x.
TARGET_ATTENTION = 2.27
MINIMUM_ATTENTION = 1.50
ATTENTION_SHARE = 0.346
QK_SHARE = 0.71
# Correctness is not a tolerance here. Each chain reduces in exactly the
# retained order over the retained operands, so anything but bit-identical
# output is a bug rather than a rounding difference.
REQUIRE_BIT_EXACT = True

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
LLAMA_DIR = str(WORK / "llama.cpp")
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ROOT = WORK / f"glcuda-h2h-{RUN_ID}"
META_REPO = ROOT / "meta"
RESULTS = ROOT / "results"
TREE = ROOT / "wave15"
TARGET = ROOT / "target-wave15"
FINAL_ZIP = WORK / "glcuda_t4_h2h_llamacpp_results.zip"
ROOT.mkdir(parents=True, exist_ok=False)
RESULTS.mkdir(parents=True)

def run(cmd, cwd=None, env=None, timeout=1800, check=True):
    merged = os.environ.copy()
    for key in [
        "GLCUDA_FORCE_Q8", "GLCUDA_GRID2D", "GLCUDA_R256", "GLCUDA_NO_MMA",
        "GLCUDA_FUSE_Q8_GLUE", "GLCUDA_GQA_GROUP", "GLCUDA_NTILE128",
        "GLCUDA_BSTAGE", "GLCUDA_TELEMETRY", "GLCUDA_CACHE", "GLCUDA_ATTN_ROWS",
        "GLCUDA_GQA7_CHAINS",
    ]:
        merged.pop(key, None)
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    p = subprocess.run(
        [str(x) for x in cmd], cwd=str(cwd) if cwd else None, env=merged,
        text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout,
    )
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}\nSTDOUT:\n{p.stdout}\nSTDERR:\n{p.stderr}")
    return p

def sh(cmd, cwd=None, timeout=7200, env=None):
    """(rc, stdout, stderr); never raises. A failing tool is data, not a lost session."""
    e = dict(os.environ)
    if env:
        e.update(env)
    try:
        # stdin=DEVNULL is not decoration: llama-cli falls into an interactive
        # loop on an inherited stdin and sits there until the timeout.
        p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True,
                           timeout=timeout, env=e, stdin=subprocess.DEVNULL)
        return p.returncode, p.stdout, p.stderr
    except subprocess.TimeoutExpired:
        return 124, "", f"timed out after {timeout}s"
    except Exception as ex:
        return 125, "", f"{type(ex).__name__}: {ex}"


def save_log(name, proc):
    (RESULTS / name).write_text(
        f"returncode={proc.returncode}\n\nSTDOUT\n{proc.stdout}\n\nSTDERR\n{proc.stderr}",
        encoding="utf-8",
    )

def archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    return FINAL_ZIP

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(8 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def fail_phase(phase):
    text = traceback.format_exc()
    (RESULTS / "FAILED.json").write_text(
        json.dumps({"phase": phase, "traceback": text}, indent=2), encoding="utf-8"
    )
    archive()
    raise RuntimeError(f"Wave 15A {phase} failed; partial archive: {FINAL_ZIP}")


def decode_patch(encoded, expected, label):
    data = gzip.decompress(base64.b64decode(encoded))
    got = hashlib.sha256(data).hexdigest()
    if got != expected:
        raise RuntimeError(f"{label} patch hash mismatch: {got} != {expected}")
    path = RESULTS / f"{label}.patch"
    path.write_bytes(data)
    return path

def ensure_cargo():
    cargo_home = WORK / ".wave15-cargo"
    rustup_home = WORK / ".wave15-rustup"
    candidates = [
        shutil.which("cargo"), cargo_home / "bin/cargo", Path.home() / ".cargo/bin/cargo",
        Path("/usr/local/cargo/bin/cargo"),
    ]
    cargo = next((Path(x) for x in candidates if x and Path(x).is_file()), None)
    if cargo is None:
        installer = ROOT / "rustup-init"
        url = "https://static.rust-lang.org/rustup/dist/x86_64-unknown-linux-gnu/rustup-init"
        last_error = None
        for attempt in range(1, 6):
            try:
                request = urllib.request.Request(
                    url, headers={"User-Agent": "GwenLand-glcuda-Wave15/1.0", "Accept-Encoding": "identity"},
                )
                with urllib.request.urlopen(request, timeout=120) as response:
                    payload = response.read()
                if len(payload) < (1 << 20):
                    raise RuntimeError(f"rustup-init download is unexpectedly small: {len(payload)} bytes")
                installer.write_bytes(payload)
                installer.chmod(0o755)
                break
            except Exception as exc:
                last_error = exc
                if attempt == 5:
                    raise RuntimeError(f"cannot download rustup-init: {last_error}") from exc
                time.sleep(min(30, 2 ** attempt))
        install_env = {
            "CARGO_HOME": str(cargo_home), "RUSTUP_HOME": str(rustup_home),
            "PATH": f"{cargo_home / 'bin'}:{os.environ.get('PATH', '')}",
        }
        install = run(
            [installer, "-y", "--profile", "minimal", "--default-toolchain", "stable", "--no-modify-path"],
            env=install_env, timeout=1800,
        )
        save_log("rustup-init.log", install)
        cargo = cargo_home / "bin/cargo"
    cargo_bin = cargo.parent
    os.environ["PATH"] = f"{cargo_bin}:{os.environ.get('PATH', '')}"
    if cargo_home in cargo.parents:
        os.environ["CARGO_HOME"] = str(cargo_home)
        os.environ["RUSTUP_HOME"] = str(rustup_home)
    cargo = Path(shutil.which("cargo") or cargo)
    rustc = shutil.which("rustc")
    if not cargo.is_file() or not rustc:
        raise RuntimeError(f"Rust toolchain bootstrap incomplete: cargo={cargo}, rustc={rustc}")
    cargo_version = run([cargo, "--version"])
    rustc_version = run([rustc, "--version", "--verbose"])
    save_log("rust-toolchain.log", cargo_version)
    with (RESULTS / "rust-toolchain.log").open("a", encoding="utf-8") as f:
        f.write(f"\n\nRUSTC\n{rustc_version.stdout}\n{rustc_version.stderr}")
    print(f"Rust toolchain: {cargo_version.stdout.strip()} / {rustc_version.stdout.splitlines()[0]}")
    return str(cargo)

PATCHES = {
    name: decode_patch(b64, sha, name)
    for name, b64, sha in [
        ("wave3", WAVE3_PATCH_GZIP_B64, WAVE3_PATCH_SHA256),
        ("wave4", WAVE4_PATCH_GZIP_B64, WAVE4_PATCH_SHA256),
        ("wave11", WAVE11_PATCH_GZIP_B64, WAVE11_PATCH_SHA256),
        ("wave12", WAVE12_PATCH_GZIP_B64, WAVE12_PATCH_SHA256),
        ("wave13", WAVE13_PATCH_GZIP_B64, WAVE13_PATCH_SHA256),
        ("wave13b", WAVE13B_PATCH_GZIP_B64, WAVE13B_PATCH_SHA256),
        ("wave15", WAVE15_PATCH_GZIP_B64, WAVE15_PATCH_SHA256),
    ]
}

gpu = run(
    ["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version",
     "--format=csv,noheader,nounits"], timeout=60,
)
rows = [x.strip() for x in gpu.stdout.splitlines() if x.strip()]
fields = [x.strip() for x in rows[0].split(",")] if rows else []
if len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
    raise RuntimeError(f"Wave 15A screening requires Tesla T4 sm_75, got {rows}")
GPU_INFO = {"raw": rows[0], "name": fields[1], "compute_cap": fields[2], "driver": fields[4]}
(RESULTS / "gpu.json").write_text(json.dumps(GPU_INFO, indent=2), encoding="utf-8")
CARGO = ensure_cargo()

run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, META_REPO], timeout=1800)
if run(["git", "cat-file", "-t", BASE_REV], cwd=META_REPO, check=False).returncode:
    run(["git", "fetch", "--depth", "1", "origin", BASE_REV], cwd=META_REPO, timeout=1800)
run(["git", "worktree", "add", "--detach", TREE, BASE_REV], cwd=META_REPO)
for name in ["wave3", "wave4", "wave11", "wave12", "wave13", "wave13b", "wave15"]:
    run(["git", "apply", "--whitespace=error", PATCHES[name]], cwd=TREE)
run(["git", "diff", "--check"], cwd=TREE)

# Wave 14 was an audit and touched nothing. Wave 15A is engine code, so this
# contract is widened deliberately rather than by accident: the new kernel must
# be present AND the retained one must still be there untouched, because the
# entire measurement is a ratio between the two.
def rust_code_only(text):
    return re.sub(r"//.*", "", text)

runner_rs = rust_code_only((TREE / "glcuda/src/runner.rs").read_text(encoding="utf-8"))
ptx = (TREE / "glcuda/src/kernels/glcuda.ptx").read_text(encoding="utf-8")
kernels_rs = rust_code_only((TREE / "glcuda/src/kernels/mod.rs").read_text(encoding="utf-8"))
attention_rs = rust_code_only((TREE / "glcuda/src/attention/mod.rs").read_text(encoding="utf-8"))
STRUCTURAL = {
    "screen_diagnostic_present": (TREE / "glcuda/examples/wave15_screen.rs").is_file(),
    "qk4_entry_present": ".visible .entry gl_attn_rows_qk4_f32(" in ptx,
    "retained_entry_still_present": ".visible .entry gl_attn_decode_rows_f32(" in ptx,
    "four_chains_in_ptx": ptx.count("fma.rn.f32 %f_a") == 4,
    "reduction_interleaved": ptx.count("shfl.sync.down.b32 %r_r") == 20,
    "qk4_wired": "attn_rows_qk4" in kernels_rs and "Qk4" in attention_rs,
    "qk4_is_the_default": "ENAttentionPath::Qk4\n    }" in attention_rs
        or "        ENAttentionPath::Qk4" in attention_rs,
    "rows_is_the_escape_hatch": "GLCUDA_ATTN_ROWS" in kernels_rs
        and "rows_forced" in attention_rs,
    "old_opt_in_flag_is_gone": "GLCUDA_QK4" not in kernels_rs,
    "dispatch_is_announced": "[glcuda-attn]" in attention_rs,
    "retained_13b_present": "qkv_stacked" in runner_rs,
    "retained_13a_present": "attention::prefill(" in runner_rs,
}
if not all(STRUCTURAL.values()):
    raise RuntimeError(f"H2H structural contract failed: {STRUCTURAL}")

p = run([CARGO, "build", "--release", "-p", "glbench", "--locked"],
        cwd=TREE, env={"CARGO_TARGET_DIR": str(TARGET)}, timeout=7200)
save_log("cargo-build-glbench.log", p)
GLBENCH = str(TARGET / "release/glbench")

p = run([CARGO, "test", "-p", "glcuda", "--lib", "--locked"],
        cwd=TREE, env={"CARGO_TARGET_DIR": str(TARGET)}, timeout=3600)
save_log("cargo-lib-tests.log", p)
LIB_TESTS = next((l for l in (p.stdout + p.stderr).splitlines()
                  if l.startswith("test result:")), "")

# Correctness gate, on the device, before a single number is read. parity.rs
# SKIPS silently without CUDA, which is exactly why the wave's own test is
# checked by name here instead of trusting a green summary line.
# --test-threads=1 is not optional: backend_buffer_returns_vram_exactly reads
# free VRAM before and after its own alloc/free cycle, so any test allocating
# concurrently reads as a leak. parity.rs says so in its module doc, and the
# first Wave 15A run failed exactly there - a harness bug, not a kernel one.
p = run([CARGO, "test", "--release", "-p", "glcuda", "--test", "parity", "--locked",
         "--", "--nocapture", "--test-threads=1"],
        cwd=TREE, env={"CARGO_TARGET_DIR": str(TARGET), "CUDA_VISIBLE_DEVICES": "0"},
        timeout=7200, check=False)
save_log("cargo-test-parity.log", p)
PARITY = {
    "returncode": p.returncode,
    "bit_exact_test_ran":
        "wave15_four_chain_qk_is_bit_exact_to_the_retained_attention ... ok" in p.stdout,
    "default_path_test_ran":
        "wave15a_four_chain_qk_is_the_default_path ... ok" in p.stdout,
    "summary": next((l for l in p.stdout.splitlines() if l.startswith("test result:")), ""),
}
if p.returncode or not PARITY["bit_exact_test_ran"] or not PARITY["default_path_test_ran"]:
    raise RuntimeError(f"Wave 15A parity gate failed: {PARITY}")

STACK_OK = True
MODEL_OK = False
print(json.dumps({"gpu": GPU_INFO, "structural": STRUCTURAL, "parity": PARITY,
                  "lib_tests": LIB_TESTS}, indent=2))


## 2 - Resumable pinned model fetch


In [ ]:
if not globals().get("STACK_OK"):
    raise RuntimeError("Bootstrap gate did not pass")

def fetch_pinned_model():
    model = WORK / HF_FILENAME
    part = WORK / f"{HF_FILENAME}.part"
    if model.is_file() and model.stat().st_size == HF_EXPECTED_BYTES and sha256_file(model) == HF_EXPECTED_SHA256:
        return model
    if model.exists():
        model.unlink()
    if part.exists():
        part_size = part.stat().st_size
        if part_size == HF_EXPECTED_BYTES:
            if sha256_file(part) == HF_EXPECTED_SHA256:
                part.replace(model)
                return model
            part.unlink()
        elif part_size > HF_EXPECTED_BYTES:
            part.unlink()
    url = f"https://huggingface.co/{HF_REPO}/resolve/{HF_REVISION}/{HF_FILENAME}?download=true"
    for attempt in range(1, 6):
        start = part.stat().st_size if part.exists() else 0
        headers = {"User-Agent": "GwenLand-glcuda-Wave12/1.0", "Accept-Encoding": "identity"}
        if start:
            headers["Range"] = f"bytes={start}-"
        request = urllib.request.Request(url, headers=headers)
        try:
            response = urllib.request.urlopen(request, timeout=120)
            status = getattr(response, "status", response.getcode())
            if start and status != 206:
                response.close()
                part.unlink(missing_ok=True)
                start = 0
                response = urllib.request.urlopen(
                    urllib.request.Request(url, headers={"User-Agent": "GwenLand-glcuda-Wave12/1.0", "Accept-Encoding": "identity"}),
                    timeout=120,
                )
                status = getattr(response, "status", response.getcode())
            if status not in (200, 206):
                raise RuntimeError(f"HTTP {status}")
            mode = "ab" if start and status == 206 else "wb"
            downloaded = start
            last_print = time.monotonic()
            with response, part.open(mode) as output:
                while True:
                    chunk = response.read(8 << 20)
                    if not chunk:
                        break
                    output.write(chunk)
                    downloaded += len(chunk)
                    if time.monotonic() - last_print >= 20:
                        print(f"model fetch {downloaded / (1 << 20):.1f}/{HF_EXPECTED_BYTES / (1 << 20):.1f} MiB")
                        last_print = time.monotonic()
            if part.stat().st_size != HF_EXPECTED_BYTES:
                raise RuntimeError(f"truncated model: {part.stat().st_size}/{HF_EXPECTED_BYTES}")
            digest = sha256_file(part)
            if digest != HF_EXPECTED_SHA256:
                part.unlink(missing_ok=True)
                raise RuntimeError(f"model SHA mismatch: {digest}")
            part.replace(model)
            return model
        except Exception as exc:
            print(f"fetch attempt {attempt}/5 failed: {exc}")
            if attempt == 5:
                raise
            time.sleep(min(30, 2 ** attempt))

try:
    MODEL_PATH = fetch_pinned_model()
    MODEL_META = {"repo": HF_REPO, "revision": HF_REVISION, "filename": HF_FILENAME, "bytes": MODEL_PATH.stat().st_size, "sha256": sha256_file(MODEL_PATH)}
    (RESULTS / "model.json").write_text(json.dumps(MODEL_META, indent=2), encoding="utf-8")
    MODEL_OK = True
    print(json.dumps(MODEL_META, indent=2))
except Exception:
    fail_phase("model-fetch")

## 3 - Build llama.cpp with CUDA (~30 min cold)


In [ ]:
if not os.path.isdir(os.path.join(LLAMA_DIR, ".git")):
    rc, o, e = sh(["git", "clone", "--depth", "1", LLAMA_REPO, LLAMA_DIR], timeout=1800)
    print((o or e)[-800:])

if LLAMA_PIN:
    rc, o, _ = sh(["git", "rev-parse", "--short", "HEAD"], cwd=LLAMA_DIR)
    if not o.strip().startswith(LLAMA_PIN[:7]):
        # A shallow clone has only the tip, so the pinned commit has to be
        # fetched explicitly before it can be checked out.
        rc, o, e = sh(["git", "fetch", "--depth", "1", "origin", LLAMA_PIN],
                      cwd=LLAMA_DIR, timeout=900)
        rc2, _, e2 = sh(["git", "checkout", "-q", LLAMA_PIN], cwd=LLAMA_DIR)
        if rc2 != 0:
            print(f"could not pin to {LLAMA_PIN} ({(e or e2).strip()[:200]}); "
                  f"staying on the default branch tip")

rc, o, _ = sh(["git", "rev-parse", "--short", "HEAD"], cwd=LLAMA_DIR)
LLAMA_COMMIT = o.strip()
print("llama.cpp commit:", LLAMA_COMMIT,
      f"(pinned to {LLAMA_PIN})" if LLAMA_PIN else "(tracking default branch)")

os.system("which ccache >/dev/null 2>&1 || (apt-get -qq install -y ccache >/dev/null 2>&1)")
USE_CCACHE = shutil.which("ccache") is not None
# ~/.ccache does NOT survive a Kaggle session. /kaggle/working does, once the
# notebook is saved, so point ccache there or every session pays full price.
os.environ["CCACHE_DIR"] = os.path.join(WORK, ".ccache")
os.environ["CCACHE_MAXSIZE"] = "5G"
os.makedirs(os.environ["CCACHE_DIR"], exist_ok=True)
print("ccache          :", "yes" if USE_CCACHE else "no (build will be slower)")
print("ccache dir      :", os.environ["CCACHE_DIR"])

LLAMA_BIN_DIR = os.path.join(LLAMA_DIR, "build", "bin")
LLAMA_BENCH = os.path.join(LLAMA_BIN_DIR, "llama-bench")
LLAMA_CLI   = os.path.join(LLAMA_BIN_DIR, "llama-cli")

# ---- the binary cache: pay for this build once, not once per session -------
#
# A 28-minute CUDA build that has to be repeated every session is the reason
# this notebook kept dying overnight. Finished binaries are copied into
# /kaggle/working, which survives a saved session, and reused next time.
#
# The manifest is what makes reuse safe: binaries built with GGML_CUDA_NO_VMM
# or without FlashAttention are NOT interchangeable with a stock build, and
# silently reusing them would put a handicap into Section 2 that nothing in
# the report could explain. Reuse only on an exact match of commit and flags.
BIN_CACHE = os.path.join(WORK, "llama-bin-cache")
CACHE_MANIFEST = os.path.join(BIN_CACHE, "manifest.json")

def cache_key():
    return {"commit": LLAMA_COMMIT, "arch": CUDA_ARCH, "fast_build": FAST_BUILD,
            "strategy": None}   # strategy filled after configure

def try_restore_cache():
    if not os.path.exists(CACHE_MANIFEST):
        return None
    try:
        man = json.load(open(CACHE_MANIFEST, encoding="utf-8"))
    except Exception:
        return None
    want = cache_key()
    if man.get("commit") != want["commit"] or man.get("arch") != want["arch"] \
            or man.get("fast_build") != want["fast_build"]:
        print(f"binary cache present but does not match "
              f"(cached: {man.get('commit')}/fa_off={man.get('fast_build')}); ignoring")
        return None
    ok = True
    os.makedirs(LLAMA_BIN_DIR, exist_ok=True)
    for name in ("llama-bench", "llama-cli"):
        src = os.path.join(BIN_CACHE, name)
        if os.path.exists(src):
            dst = os.path.join(LLAMA_BIN_DIR, name)
            shutil.copy2(src, dst)
            os.chmod(dst, 0o755)
        elif name == "llama-bench":
            ok = False
    # Shared libraries the tools link against travel with them.
    for so in glob.glob(os.path.join(BIN_CACHE, "*.so*")):
        shutil.copy2(so, os.path.join(LLAMA_BIN_DIR, os.path.basename(so)))
    return man if ok else None

def save_cache(strategy):
    os.makedirs(BIN_CACHE, exist_ok=True)
    saved = []
    for name in ("llama-bench", "llama-cli"):
        src = os.path.join(LLAMA_BIN_DIR, name)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(BIN_CACHE, name))
            saved.append(name)
    for so in glob.glob(os.path.join(LLAMA_BIN_DIR, "*.so*")):
        shutil.copy2(so, os.path.join(BIN_CACHE, os.path.basename(so)))
    man = cache_key()
    man["strategy"] = strategy
    man["saved"] = saved
    json.dump(man, open(CACHE_MANIFEST, "w", encoding="utf-8"), indent=1)
    print(f"binary cache written to {BIN_CACHE}: {saved}")
    print("  Save this notebook version so /kaggle/working persists, then the "
          "next run skips the build entirely.")

RESTORED = try_restore_cache()

# Cold vs warm, decided before the timer -- a rebuilt tree relinks in seconds
# and that number is not a build time.
LLAMA_COLD = not os.path.exists(LLAMA_BENCH)

# ggml-cuda links CUDA::cuda_driver, a target FindCUDAToolkit only creates if
# it can find libcuda.so. A CUDA image can ship the toolkit without the
# development symlink, so the toolkit is "found" and the target still is not:
#
#   CMake Error at ggml/src/ggml-cuda/CMakeLists.txt:182
#     Target "ggml-cuda" links to: CUDA::cuda_driver
#     but the target was not found.
#
# Find the library ourselves and hand CMake the path.
LIBCUDA_CANDIDATES = []

# 1. The dynamic linker cache. Authoritative and path-independent -- a glob
#    list only finds libcuda where you already guessed it might be, and on
#    this image the first version of that guess found nothing at all.
rc, o, _ = sh(["ldconfig", "-p"], timeout=120)
for line in o.splitlines():
    if "libcuda.so" in line and "=>" in line:
        p = line.split("=>")[-1].strip()
        if os.path.exists(p):
            LIBCUDA_CANDIDATES.append(p)

# 2. Toolkit stubs, which ldconfig never lists because they are link-time only.
for pat in ["/usr/local/cuda*/lib64/stubs/libcuda.so",
            "/usr/local/cuda*/targets/*/lib/stubs/libcuda.so",
            "/usr/local/cuda/lib64/stubs/libcuda.so"]:
    LIBCUDA_CANDIDATES += sorted(glob.glob(pat))

# 3. A bounded sweep, so a failure produces evidence rather than a shrug.
rc, o, _ = sh(["bash", "-lc",
               "find /usr /opt /lib /lib64 -maxdepth 6 -name 'libcuda.so*' "
               "2>/dev/null | head -40"], timeout=300)
LIBCUDA_CANDIDATES += [ln.strip() for ln in o.splitlines() if ln.strip()]

# Dedupe, preferring the link-time name `libcuda.so` over a runtime `.so.N`.
seen, ordered = set(), []
for p in LIBCUDA_CANDIDATES:
    if p not in seen:
        seen.add(p)
        ordered.append(p)
ordered.sort(key=lambda p: (os.path.basename(p) != "libcuda.so", len(p)))
LIBCUDA_CANDIDATES = ordered
print("libcuda candidates:")
for p in LIBCUDA_CANDIDATES:
    print("   ", p)
if not LIBCUDA_CANDIDATES:
    print("    NONE FOUND -- the NO_VMM fallback will be needed")

LIBCUDA = LIBCUDA_CANDIDATES[0] if LIBCUDA_CANDIDATES else None

# find_library looks for the bare `libcuda.so` name. If all we have is a
# runtime `libcuda.so.1`, give CMake a directory where that name exists.
LIBCUDA_SHIM = None
if LIBCUDA and os.path.basename(LIBCUDA) != "libcuda.so":
    LIBCUDA_SHIM = os.path.join(WORK, "cuda-link-shim")
    os.makedirs(LIBCUDA_SHIM, exist_ok=True)
    link = os.path.join(LIBCUDA_SHIM, "libcuda.so")
    if not os.path.exists(link):
        try:
            os.symlink(LIBCUDA, link)
        except OSError as ex:
            print(f"  could not create link shim: {ex}")
            LIBCUDA_SHIM = None
    if LIBCUDA_SHIM:
        print(f"  link shim: {link} -> {LIBCUDA}")

BASE_CFG = ["cmake", "-B", "build",
            "-DGGML_CUDA=ON",
            "-DCMAKE_BUILD_TYPE=Release",
            f"-DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH}",
            "-DLLAMA_CURL=OFF",
            "-DLLAMA_BUILD_TESTS=OFF",
            "-DLLAMA_BUILD_EXAMPLES=OFF",
            # ⛔ NOT OFF, however much it looks like a saving. In tools/CMakeLists.txt:
            #
            #     if (LLAMA_BUILD_SERVER)
            #         add_subdirectory(ui)
            #         add_subdirectory(cli)
            #         add_subdirectory(server)
            #     endif()
            #
            # `llama-cli` moved under the server gate (tools/main -> tools/cli),
            # so -DLLAMA_BUILD_SERVER=OFF deletes the CLI target and the build
            # ends with "No rule to make target 'llama-cli'" after 28 minutes of
            # compiling everything else. `--target` still limits what is
            # actually compiled, so turning this ON costs configure time, not
            # build time.
            "-DLLAMA_BUILD_SERVER=ON",
            # Multi-GPU collectives. Safe here only because the config cell
            # pins CUDA_VISIBLE_DEVICES=0 for both tools -- an earlier version
            # of this comment claimed no second card was in play, which was
            # simply wrong: Kaggle gives two T4s and llama.cpp was observed
            # using both. With one device enforced, this cannot change
            # throughput, unlike NO_VMM or FA.
            "-DGGML_CUDA_NCCL=OFF"]
if FAST_BUILD:
    BASE_CFG.append("-DGGML_CUDA_FA=OFF")
if USE_CCACHE:
    BASE_CFG += ["-DCMAKE_C_COMPILER_LAUNCHER=ccache",
                 "-DCMAKE_CXX_COMPILER_LAUNCHER=ccache",
                 "-DCMAKE_CUDA_COMPILER_LAUNCHER=ccache"]

# Ordered by how little they change about llama.cpp. GGML_CUDA_NO_VMM is last
# on purpose: upstream's own comment says it exists to avoid linking the
# driver lib, so it WILL configure -- but it also turns off the VMM pool
# allocator, which is a performance-relevant change to the tool being
# measured. Taking it silently would quietly handicap llama.cpp in Section 2.
ATTEMPTS = []
if LIBCUDA:
    _dirs = [d for d in [LIBCUDA_SHIM, os.path.dirname(LIBCUDA)] if d]
    ATTEMPTS.append((f"libcuda pinned to {LIBCUDA}",
                     [f"-DCUDA_cuda_driver_LIBRARY={LIBCUDA}",
                      f"-DCUDA_CUDA_LIBRARY={LIBCUDA}",
                      "-DCMAKE_LIBRARY_PATH=" + ";".join(_dirs)]))
ATTEMPTS.append(("stock configuration", []))
ATTEMPTS.append(("GGML_CUDA_NO_VMM=ON -- VMM pool allocator DISABLED",
                 ["-DGGML_CUDA_NO_VMM=ON"]))

LLAMA_BUILD_NOTES = []
CONFIG_OK, CONFIG_STRATEGY = False, None
t0 = time.time()

if RESTORED:
    CONFIG_OK = True
    CONFIG_STRATEGY = RESTORED.get("strategy") or "restored from binary cache"
    ATTEMPTS = []
    print(f"reusing cached binaries: {RESTORED.get('saved')} "
          f"(built at {RESTORED.get('commit')}, strategy: {CONFIG_STRATEGY})")
    print("no build needed")

for label, extra in ATTEMPTS:
    # A poisoned cache makes the next attempt re-fail for the previous
    # attempt's reason. Clear it, not the whole tree -- ccache keeps the
    # object files either way.
    for stale in ["build/CMakeCache.txt", "build/CMakeFiles"]:
        p = os.path.join(LLAMA_DIR, stale)
        if os.path.isdir(p):
            shutil.rmtree(p, ignore_errors=True)
        elif os.path.exists(p):
            os.remove(p)

    print(f"\n--- configure: {label} ---")
    rc, o, e = sh(BASE_CFG + extra, cwd=LLAMA_DIR, timeout=1800)
    if rc == 0:
        CONFIG_OK, CONFIG_STRATEGY = True, label
        print("configure OK")
        break
    tail = [ln for ln in (o + e).splitlines() if "error" in ln.lower()][-6:]
    print("failed:", " | ".join(tail) if tail else (o + e)[-600:])

if not CONFIG_OK:
    print("\nALL CONFIGURE ATTEMPTS FAILED")
else:
    if "NO_VMM" in CONFIG_STRATEGY:
        LLAMA_BUILD_NOTES.append(
            "Built with `GGML_CUDA_NO_VMM=ON`. The stock configuration could "
            "not link `CUDA::cuda_driver` on this image. This disables ggml's "
            "CUDA virtual-memory pool allocator, so llama.cpp's throughput "
            "here may be BELOW what a stock build achieves. Any decode "
            "advantage glbench shows in Section 2 must be read with this in "
            "mind.")
    elif LIBCUDA and "pinned" in CONFIG_STRATEGY:
        LLAMA_BUILD_NOTES.append(
            f"`CUDA::cuda_driver` was not auto-discovered; libcuda was pinned "
            f"to `{LIBCUDA}`. This changes nothing about the code that runs.")

    if FAST_BUILD:
        LLAMA_BUILD_NOTES.append(
            "Built with `GGML_CUDA_FA=OFF`: ggml's CUDA FlashAttention kernels "
            "were not compiled, so llama.cpp runs without them. This was done "
            "to fit the build into the machine, not because FA is irrelevant, "
            "and it may depress llama.cpp's numbers below a stock build.")

    rc, o, e = sh(["cmake", "--build", "build", "--config", "Release",
                   "-j", str(BUILD_JOBS),
                   "--target", "llama-bench", "llama-cli"],
                  cwd=LLAMA_DIR, timeout=21600)
    if rc != 0:
        print("BUILD FAILED"); print((o + e)[-6000:])
    else:
        save_cache(CONFIG_STRATEGY)

LLAMA_BUILD_SECS = time.time() - t0
print(f"\nllama.cpp build: {LLAMA_BUILD_SECS/60:.1f} min "
      f"[{'COLD' if LLAMA_COLD else 'WARM - a relink, not a build time'}]")
print(f"strategy: {CONFIG_STRATEGY}")
for n in LLAMA_BUILD_NOTES:
    print("NOTE:", n)


## 4 - CUDA verification: a hard gate on llama.cpp


In [ ]:
# llama-bench is the throughput comparison; without it there is nothing to
# compare. llama-cli only feeds the behavioral section and the tokenizer
# cross-check, so its absence degrades the report rather than voiding it --
# aborting the whole run over it would throw away a working Section 2.
if not os.path.exists(LLAMA_BENCH):
    raise RuntimeError(
        f"llama-bench missing ({LLAMA_BENCH}). Read the build log above. "
        "Not continuing: there is nothing to compare against."
    )

HAVE_CLI = os.path.exists(LLAMA_CLI)
if not HAVE_CLI:
    print("WARNING: llama-cli was not built. Sections 5 and the tokenizer "
          "cross-check will be reported as unavailable, and Section 1 will "
          "say so.")

# The only trustworthy probe is an actual run. A one-token bench loads the
# model through the same path the real measurement will use.
rc, smoke_out, smoke_err = sh([LLAMA_BENCH, "-m", MODEL_PATH,
                               "-p", "1", "-n", "1", "-r", "1", "-ngl", "99"],
                              timeout=900)
SMOKE = smoke_out + "\n" + smoke_err
CUDA_MARKERS = ["ggml_cuda_init", "CUDA0", "using CUDA", "found 1 CUDA", "found 2 CUDA"]
LLAMA_CUDA = any(m in SMOKE for m in CUDA_MARKERS)

print(SMOKE[-2500:])
print("\nCUDA backend detected:", LLAMA_CUDA)

if not LLAMA_CUDA:
    raise RuntimeError(
        "llama.cpp is running WITHOUT CUDA.\n"
        "Refusing to continue (D3): a CPU llama.cpp against a CUDA glbench "
        "produces a throughput table that looks like a result and is not one.\n"
        "Fix the build (check nvcc is on PATH and GGML_CUDA=ON took effect), "
        "or state the CPU fallback explicitly in Section 1 and re-run with this "
        "guard removed on purpose."
    )


# Reaching here means llama.cpp is built AND CUDA-enabled.
LLAMA_OK = True
print("llama.cpp ready:", LLAMA_BENCH)


## 5 - Interleaved head-to-head


In [ ]:
if not globals().get("MODEL_OK"):
    raise RuntimeError("Model gate did not pass")
if not globals().get("LLAMA_OK"):
    raise RuntimeError("llama.cpp build gate did not pass")

import math

prompt_unit = "Measure this deterministic systems prompt carefully. Explain how token-parallel integer matrix multiplication uses shared memory, Tensor Cores, and fixed launch geometry. "
FIXED_PROMPT = prompt_unit * 8

# glcuda's PRODUCTION stack. The 2026-08-21 comparison set none of this, which
# is the single reason its 5.85x is not an engine comparison.
GL_ENV = {
    "CUDA_VISIBLE_DEVICES": "0",
    "GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1", "GLCUDA_FUSE_Q8_GLUE": "1",
    "GLCUDA_NTILE128": "1", "GLCUDA_BSTAGE": "1",
}
# Kaggle hands out two T4s and llama.cpp will split a model across both, which
# would compare card counts rather than engines.
LB_ENV = {"CUDA_VISIBLE_DEVICES": "0"}

def percentile(values, q):
    values = sorted(values)
    i = (len(values) - 1) * q
    lo, hi = math.floor(i), math.ceil(i)
    return values[lo] if lo == hi else values[lo] * (hi - i) + values[hi] * (i - lo)

def run_glcuda(out, cold, warmup, iters):
    cmd = [GLBENCH, "run", "--engine", "glcuda", "--model", str(MODEL_PATH),
           "--prompt", FIXED_PROMPT, "--tokens", "1",
           "--cold-iters", str(cold), "--warmup", str(warmup), "--iters", str(iters),
           "--temperature", "0", "--seed", "42", "--kind", "prefill",
           "--verify-against", "glproc", "--out", str(out)]
    return run(cmd, cwd=TREE, env=GL_ENV, timeout=14400, check=False)

def glcuda_stats(path):
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    validation = data.get("validation") or {}
    if validation.get("passed") is not True:
        raise RuntimeError(f"glcuda session validation failed: {validation}")
    parity = [f for f in validation.get("findings", []) if f.get("check") == "parity"]
    m = re.search(r"(\d+)/(\d+) tokens match oracle", parity[-1].get("message", "")) if parity else None
    if not m or m.group(1) != m.group(2):
        raise RuntimeError("glcuda did not match the glproc oracle exactly")
    it = (data.get("measurements") or {}).get("iterations") or []
    counts = {int(x["prompt_tokens"]) for x in it}
    if len(counts) != 1:
        raise RuntimeError(f"ragged prompt_tokens: {counts}")
    ms = [float(x["prefill_ms"]) for x in it]
    ntok = counts.pop()   # hoisted: inside the comprehension this empties
    tps = [ntok * 1000.0 / x for x in ms]   # the set on iteration 2
    return {"p50": percentile(tps, .5), "mean": statistics.mean(tps),
            "median_ms": statistics.median(ms), "max_ms": max(ms), "oracle": "exact"}

# Self-test the stats path before spending an hour on it. Run 3 reached
# minute 60 and then died here on a two-iteration input; one iteration (the
# probe) was not enough to expose it.
_fake = {"validation": {"passed": True, "findings":
             [{"check": "parity", "message": "4/4 tokens match oracle"}]},
         "measurements": {"iterations": [
             {"prompt_tokens": 100, "prefill_ms": 50.0},
             {"prompt_tokens": 100, "prefill_ms": 100.0}]}}
_sp = RESULTS / "_selftest.json"
_sp.write_text(json.dumps(_fake), encoding="utf-8")
assert abs(glcuda_stats(_sp)["mean"] - 1500.0) < 1e-6, "glcuda_stats arithmetic"
_sp.unlink()

def run_llamacpp(tokens, iters, tag):
    """One llama-bench prompt-processing run.

    Every invocation dumps stdout and stderr into RESULTS. Two runs have now
    died where the only evidence was the Kaggle kernel log, and that endpoint
    was rate-limited for an hour afterwards - so the evidence has to travel
    with the results archive instead.
    """
    cmd = [LLAMA_BENCH, "-m", str(MODEL_PATH), "-p", str(tokens), "-n", "0",
           "-r", str(iters), "-ngl", "99", "-o", "json"]
    rc, out, err = sh(cmd, env=LB_ENV, timeout=14400)
    (RESULTS / f"llamabench-{tag}.log").write_text(
        f"$ {' '.join(str(c) for c in cmd)}\nreturncode={rc}\n\n"
        f"STDOUT\n{out}\n\nSTDERR\n{err}",
        encoding="utf-8",
    )
    if rc:
        raise RuntimeError(f"llama-bench failed ({rc}); see llamabench-{tag}.log")
    text = out.strip()
    if not text.startswith("["):
        m = re.search(r"\[\s*\{.*\}\s*\]", out, re.S)
        if not m:
            raise RuntimeError(
                f"llama-bench produced no JSON array; see llamabench-{tag}.log "
                f"(stdout began {out[:200]!r})"
            )
        text = m.group(0)
    payload = json.loads(text)
    rows = [r for r in payload if int(r.get("n_prompt", 0)) == tokens]
    if not rows:
        got = [(r.get("n_prompt"), r.get("n_gen")) for r in payload]
        raise RuntimeError(f"no pp row for {tokens} tokens; rows were {got}")
    row = rows[-1]
    for field in ("avg_ts", "n_prompt"):
        if field not in row:
            raise RuntimeError(f"llama-bench row is missing {field!r}: {sorted(row)}")
    return {"tps": float(row["avg_ts"]), "stddev": float(row.get("stddev_ts", 0.0)),
            "raw": row}

# One glcuda run first, only to learn the token count llama.cpp must match.
p = run_glcuda(RESULTS / "probe.json", 0, 1, 1)
save_log("glcuda-probe.log", p)
if p.returncode:
    raise RuntimeError("glcuda probe run failed")
CONTRACT = json.loads(re.findall(r"\[glcuda-contract\]\s*(\{[^\n]+\})", p.stdout + p.stderr)[-1])
EXPECTED = {"exact_fusion": True, "grid2d": True, "ntile128": True, "bstage": True}
if {k: CONTRACT.get(k) for k in EXPECTED} != EXPECTED:
    raise RuntimeError(f"glcuda is NOT in its production configuration: {CONTRACT}")
# Smoke the llama.cpp side before committing to the measurement loop. The
# previous two runs each spent ~66 minutes to discover a contract mismatch in
# the last cell; this finds one in seconds.
SMOKE = run_llamacpp(8, 1, "smoke")
print(f"llama-bench smoke: {SMOKE['tps']:.1f} tok/s at 8 prompt tokens")

PROMPT_TOKENS = int(json.loads((RESULTS / "probe.json").read_text(encoding="utf-8"))
                    ["measurements"]["iterations"][0]["prompt_tokens"])
print(f"prompt is {PROMPT_TOKENS} tokens; llama-bench will be told -p {PROMPT_TOKENS}")

ORDERS = [["glcuda", "llamacpp"], ["llamacpp", "glcuda"],
          ["glcuda", "llamacpp"], ["llamacpp", "glcuda"]]
assert len(ORDERS) == PRODUCTION_REPEATS
assert all(sum(o.index(a) for o in ORDERS) == 2 for a in ("glcuda", "llamacpp"))

RECORDS = []
for repeat, order in enumerate(ORDERS):
    for position, tool in enumerate(order):
        if tool == "glcuda":
            out = RESULTS / f"glcuda-{repeat}-{position}.json"
            q = run_glcuda(out, COLD_ITERS, WARMUP_ITERS, MEASURE_ITERS)
            save_log(f"glcuda-{repeat}-{position}.log", q)
            if q.returncode:
                raise RuntimeError(f"glcuda failed at repeat {repeat}")
            st = glcuda_stats(out)
            RECORDS.append({"repeat": repeat, "position": position, "tool": tool,
                            "tps": st["p50"], **st})
        else:
            st = run_llamacpp(PROMPT_TOKENS, MEASURE_ITERS, f"{repeat}-{position}")
            (RESULTS / f"llamacpp-{repeat}-{position}.json").write_text(
                json.dumps(st["raw"], indent=2), encoding="utf-8")
            RECORDS.append({"repeat": repeat, "position": position, "tool": tool,
                            "tps": st["tps"], "stddev": st["stddev"]})
        print(f"  {tool:9s} r{repeat} p{position}: {RECORDS[-1]['tps']:8.1f} tok/s")

SUMMARY = {}
for tool in ("glcuda", "llamacpp"):
    rows = [r for r in RECORDS if r["tool"] == tool]
    SUMMARY[tool] = {
        "median_tps": statistics.median(r["tps"] for r in rows),
        "min_tps": min(r["tps"] for r in rows),
        "max_tps": max(r["tps"] for r in rows),
        "spread": max(r["tps"] for r in rows) / min(r["tps"] for r in rows) - 1,
        "positions": [r["position"] for r in rows],
    }
PAIRED = []
for repeat in range(PRODUCTION_REPEATS):
    g = next(r for r in RECORDS if r["repeat"] == repeat and r["tool"] == "glcuda")
    l = next(r for r in RECORDS if r["repeat"] == repeat and r["tool"] == "llamacpp")
    PAIRED.append({"repeat": repeat, "ratio": l["tps"] / g["tps"]})

RATIO = SUMMARY["llamacpp"]["median_tps"] / SUMMARY["glcuda"]["median_tps"]
WORST = max(p["ratio"] for p in PAIRED)
BEST = min(p["ratio"] for p in PAIRED)
if RATIO <= 1.05:
    VERDICT = (f"AT PARITY - llama.cpp is {RATIO:.2f}x, inside the ~8% this machine "
               "drifts between sessions. The 5.85x was an artifact of comparing "
               "their tuned path against ours with tuning switched off.")
elif RATIO <= 1.5:
    VERDICT = (f"BEHIND BY {RATIO:.2f}x - real but far from the 5.85x on record, and "
               "small enough that the quantisation asymmetry below may account for "
               "part of it.")
else:
    VERDICT = (f"BEHIND BY {RATIO:.2f}x - a real gap worth a target, though still "
               f"{STALE_CLAIM_RATIO / RATIO:.1f}x smaller than the figure on record.")

lines = [
    "# glcuda vs llama.cpp - the comparison the 5.85x figure should have been", "",
    f"- notebook: {NOTEBOOK_BUILD}",
    f"- GPU: {GPU_INFO['raw']}",
    f"- patch: {WAVE15_PATCH_SHA256}",
    f"- model SHA-256: {MODEL_META['sha256']}",
    f"- parity gate: {PARITY['summary']}",
    f"- prompt: {PROMPT_TOKENS} tokens, {PRODUCTION_REPEATS} repeats, position-balanced",
    "",
    "## What the figure on record actually measured",
    "",
    f"The 2026-08-21 run reported llama.cpp **{STALE_CLAIM_RATIO}x** ahead at prefill.",
    f"Its notebook sets **no optimisation flags**, so it measured glcuda at",
    f"**{STALE_CLAIM_GLCUDA_TPS:.0f} tok/s** - while Wave 11, in the same era and with",
    "the production stack, was already at 7397. It compared llama.cpp's tuned",
    "path against ours with the tuning switched off.",
    "",
    "This run sets the production stack and asserts it from the engine's own",
    "dispatch line before measuring:",
    "",
    "```",
    json.dumps(CONTRACT, indent=2),
    "```",
    "",
    "## Measured, interleaved",
    "",
    "| tool | median tok/s | min | max | spread | positions |",
    "|---|---:|---:|---:|---:|---|",
]
for tool in ("glcuda", "llamacpp"):
    s = SUMMARY[tool]
    lines.append(
        f"| `{tool}` | {s['median_tps']:.1f} | {s['min_tps']:.1f} | {s['max_tps']:.1f} |"
        f" {s['spread'] * 100:.1f}% | {s['positions']} |"
    )
lines += [
    "",
    "| repeat | llama.cpp / glcuda |",
    "|---:|---:|",
]
for p_ in PAIRED:
    lines.append(f"| {p_['repeat']} | {p_['ratio']:.3f}x |")
lines += [
    "",
    f"- median ratio **{RATIO:.3f}x**, best repeat {BEST:.3f}x, worst {WORST:.3f}x",
    f"- **{VERDICT}**",
    "",
    "## The asymmetry that remains, stated rather than buried",
    "",
    "llama.cpp runs the model as **native Q4_K**. glcuda runs with",
    "`GLCUDA_FORCE_Q8`, which repacks the weights to **Q8_0** - twice the weight",
    "bytes. That is a real handicap on the glcuda side and it is not corrected",
    "here, because removing it would change which kernels run and stop this",
    "being a comparison of the paths each engine actually ships.",
    "",
    "Read the ratio as *engine as shipped vs engine as shipped*, not as a",
    "like-for-like kernel comparison. Wave 14 did measure that the FFN GEMM is",
    "not bandwidth-bound at this scale (-0.9% across the L2 crossing), which",
    "bounds how much the extra bytes can be costing.",
    "",
    "## Method notes",
    "",
    "- Both tools are pinned to `CUDA_VISIBLE_DEVICES=0`. Kaggle hands out two",
    "  T4s and llama.cpp will split a model across both, which would compare",
    "  card counts rather than engines.",
    "- glcuda verifies every session against the glproc oracle exactly; a",
    "  session that does not match is an error, not a slow result.",
    "- `llama-bench -p N -n 0` is prompt processing only, which is the same work",
    "  glcuda's `--kind prefill` does. The decode comparison is deliberately",
    "  absent: `llama-bench`'s generation test does no sampling at all, so it was",
    "  never the same work.",
]
report = "\n".join(lines)
(RESULTS / "h2h.json").write_text(
    json.dumps({"records": RECORDS, "summary": SUMMARY, "paired": PAIRED,
                "ratio": RATIO, "contract": CONTRACT, "prompt_tokens": PROMPT_TOKENS,
                "verdict": VERDICT}, indent=2), encoding="utf-8")
(RESULTS / "REPORT.md").write_text(report, encoding="utf-8")
archive()
print(report)
print(f"\nDownload archive: {FINAL_ZIP} ({FINAL_ZIP.stat().st_size / 1024:.1f} KB)")
